# Bibliotheque + lecture Excel

In [1]:
import pandas as pd 
import numpy as np 
import os
import re
from collections import defaultdict
from datetime import datetime, timedelta
import datetime

In [2]:
df = pd.read_excel('AGING_Ceinture_Montre (avec groupe).xlsx',sheet_name=None)

C:\Users\judupont\anaconda3\Lib\site-packages\openpyxl\worksheet\_read_only.py:85: UserWarning: Conditional Formatting extension is not supported and will be removed
  for idx, row in parser.parse():
C:\Users\judupont\anaconda3\Lib\site-packages\openpyxl\worksheet\_read_only.py:85: UserWarning: Conditional Formatting extension is not supported and will be removed
  for idx, row in parser.parse():


In [3]:
df.keys()

dict_keys(['APHM', 'CAEN', 'POITIERS', 'ROUEN', 'LAVERAN', 'LPC', 'SAINTE-MARGUERITE', 'CGD', 'CERCA', 'Effectifs ceinture', 'Effectifs montre', 'CODEBOOK '])

# Vérification blocs 

In [4]:
 # ===== Conversion heures du Excel =====
def convertir_heure_excel_ou_texte(x):
     if pd.isna(x):
        return pd.NaT
     if isinstance(x, (int, float)):
        return pd.to_datetime(x, unit="D", origin="1899-12-30")
     try:
        return pd.to_datetime(str(x).strip(), format="%H:%M")
     except Exception:
         try:
             return pd.to_datetime(str(x).strip())
         except Exception:
            return pd.NaT

In [5]:
def analyser_feuille_bloc(
    df,
    nom_feuille,
    col_debut,
    col_fin,
    nom_bloc
):
    print(f"\n--- Feuille : {nom_feuille} | {nom_bloc} ---")

    col_id = "Numero_inclusion"

    # ===== Dictionnaire des colonnes atitrées manuellement =====
    REGLES_CANDIDATS = {
        ("CAEN", "bloc2", None): {"col_fin": "heure_rappel_cond_v4"},
        ("CAEN", "bloc3", None): {"col_debut": "heure_rappel_cond_v4"},
        ("CAEN", "bloc2", "0326BJR"): {"col_fin": "heure_fluence_sem_v4"},
        ("CAEN", "bloc3", "0326BJR"): {"col_debut": "heure_fluence_sem_v4"},

        ("POITIERS", 'bloc2', "0408BCS"): {"col_fin": "heure_rappel_cond_v4"},
        ("POITIERS", 'bloc3', "0408BCS"): {"col_debut": "heure_rappel_cond_v4"},

        ("LPC", 'bloc2', "0112DRR"): {"col_fin": "heure_rappel_cond_v4"},
        ("LPC", 'bloc3', "0112DRR"): {"col_debut": "heure_rappel_cond_v4"},
        ("LPC", 'bloc2', "0115MHR"): {"col_fin": "heure_rappel_cond_v4"},
        ("LPC", 'bloc3', "0115MHR"): {"col_debut": "heure_rappel_cond_v4"},
        ("LPC", 'bloc2', "0141HJR"): {"col_fin": "heure_rappel_cond_v4"},
        ("LPC", 'bloc3', "0141HJR"): {"col_debut": "heure_rappel_cond_v4"},
        ("LPC", 'bloc2', "0144ZGR"): {"col_fin": "heure_rappel_cond_v4"},
        ("LPC", 'bloc3', "0144ZGR"): {"col_debut": "heure_rappel_cond_v4"},

        ("CERCA", 'bloc2', "0406BBS"): {"col_fin": "heure_nback_v4"},
        ("CERCA", 'bloc3', "0406BBS"): {"col_debut": "heure_nback_v4"},
        
        ("CERCA", 'bloc2', "0408SJS"): {"col_fin": "heure_nback_v4"},
        ("CERCA", 'bloc3', "0408SJS"): {"col_debut": "heure_nback_v4"},
        
        ("CERCA", 'bloc2', "0409HCR"): {"col_fin": "heure_nback_v4"},
        ("CERCA", 'bloc3', "0409HCR"): {"col_debut": "heure_nback_v4"},
        
        ("CERCA", 'bloc2', "0410FMS"): {"col_fin": "heure_nback_v4"},
        ("CERCA", 'bloc3', "0410FMS"): {"col_debut": "heure_nback_v4"},
        
        ("CERCA", 'bloc2', "0418TJR"): {"col_fin": "heure_nback_v4"},
        ("CERCA", 'bloc3', "0418TJR"): {"col_debut": "heure_nback_v4"},
        
        ("CERCA", 'bloc2', "0420MCR"): {"col_fin": "heure_nback_v4"},
        ("CERCA", 'bloc3', "0420MCR"): {"col_debut": "heure_nback_v4"},
        
        ("CERCA", 'bloc2', "0423SVS"): {"col_fin": "heure_nback_v4"},
        ("CERCA", 'bloc3', "0423SVS"): {"col_debut": "heure_nback_v4"},
        
        ("CERCA", 'bloc2', "0432CGS"): {"col_fin": "heure_nback_v4"},
        ("CERCA", 'bloc3', "0432CGS"): {"col_debut": "heure_nback_v4"},
        
    }

    # ===== Colonnes effectives par défaut =====
    col_debut_effective = col_debut
    col_fin_effective = col_fin

    # ===== Application des règles =====
    for (centre, bloc, candidat), regle in REGLES_CANDIDATS.items():
        if centre != nom_feuille:
            continue
        if bloc is not None and bloc != nom_bloc:
            continue
        if candidat is not None:
            mask_candidat = df[col_id] == candidat
            if not mask_candidat.any():
                continue

        # Colonnes à remplacer
        if "col_debut" in regle:
            col_debut_effective = regle["col_debut"]
        if "col_fin" in regle:
            col_fin_effective = regle["col_fin"]

        print(f"🔧 Règle appliquée → centre={centre}, bloc={bloc}, candidat={candidat}")

    # ===== Vérification colonnes =====
    for col in [col_debut_effective, col_fin_effective, col_id]:
        if col not in df.columns:
            print(f"❌ Colonne manquante : {col}")
            return None

    # ===== Copie du DataFrame =====
    df = df.copy()

    # Conversion des colonnes utilisées
    df[col_debut_effective] = df[col_debut_effective].apply(convertir_heure_excel_ou_texte)
    df[col_fin_effective] = df[col_fin_effective].apply(convertir_heure_excel_ou_texte)

    # ===== Durée =====
    col_duree = f"duree_{nom_bloc}"
    col_rejet = f"rejeter_{nom_bloc}"

    df[col_duree] = (df[col_fin_effective] - df[col_debut_effective]).dt.total_seconds() / 60

    # Passage de minuit
    df.loc[df[col_duree] < 0, col_duree] += 24 * 60

    # ===== Masques =====
    mask_manquant = df[col_duree].isna()
    mask_valide = df[col_duree].notna() & (df[col_duree] > 0)

    # ===== Statistiques sur la durée =====
    moyenne = df.loc[mask_valide, col_duree].mean()
    ecart_type = df.loc[mask_valide, col_duree].std()
    print(f"Moyenne = {moyenne:.2f} min")
    print(f"Écart-type = {ecart_type:.2f} min")

    borne_inf = moyenne - 2 * ecart_type

    # ===== Filtre =====
    df[col_rejet] = 0
    df.loc[mask_manquant, col_rejet] = 2

    mask_rejet = df[col_duree] < borne_inf

    # Règle spécifique bloc 1
    if nom_bloc == "bloc1":
        mask_rejet = mask_rejet | (df[col_duree] < 10)

    df.loc[mask_valide & mask_rejet, col_rejet] = 1

    # ===== Résumé =====
    print(f"\nRésumé {col_rejet} :")
    print(df[col_rejet].value_counts().sort_index())

    return {
        "df": df,
        "moyenne": moyenne,
        "ecart_type": ecart_type,
        "rejets": {
            "manquant": df.loc[df[col_rejet] == 2, col_id].tolist(),
            "rejetes": df.loc[df[col_rejet] == 1, col_id].tolist(),
        }
    }


## Bloc 1

In [6]:
feuilles = [
    'APHM', 'CAEN', 'POITIERS', 'ROUEN',
    'LAVERAN', 'LPC', 'SAINTE-MARGUERITE',
    'CGD', 'CERCA'
]

resultats_bloc1 = {}

for feuille in feuilles:
    print("\n" + "=" * 60)

    resultats_bloc1[feuille] = analyser_feuille_bloc(
        df=df[feuille],
        nom_feuille=feuille,
        col_debut="heure_ceinture_v4",   
        col_fin="heure_anamnese_fin_v4",        
        nom_bloc="bloc1"
    )




--- Feuille : APHM | bloc1 ---
Moyenne = 20.00 min
Écart-type = 8.72 min

Résumé rejeter_bloc1 :
rejeter_bloc1
0    15
1     3
2     7
Name: count, dtype: int64


--- Feuille : CAEN | bloc1 ---
Moyenne = 13.33 min
Écart-type = 5.03 min

Résumé rejeter_bloc1 :
rejeter_bloc1
0    40
1     9
2     7
Name: count, dtype: int64


--- Feuille : POITIERS | bloc1 ---
Moyenne = 20.27 min
Écart-type = 7.04 min

Résumé rejeter_bloc1 :
rejeter_bloc1
0    11
2     4
Name: count, dtype: int64


--- Feuille : ROUEN | bloc1 ---
Moyenne = 17.13 min
Écart-type = 7.02 min

Résumé rejeter_bloc1 :
rejeter_bloc1
0    14
1     1
Name: count, dtype: int64


--- Feuille : LAVERAN | bloc1 ---
Moyenne = 21.75 min
Écart-type = 2.22 min

Résumé rejeter_bloc1 :
rejeter_bloc1
0    4
2    1
Name: count, dtype: int64


--- Feuille : LPC | bloc1 ---
Moyenne = 19.16 min
Écart-type = 7.90 min

Résumé rejeter_bloc1 :
rejeter_bloc1
0    41
1     2
2     1
Name: count, dtype: int64


--- Feuille : SAINTE-MARGUERITE | bloc1

## Bloc 2 

In [7]:
resultats_bloc2 = {}

for feuille in feuilles:
    print("\n" + "=" * 60)

    resultats_bloc2[feuille] = analyser_feuille_bloc(
        df=df[feuille],
        nom_feuille=feuille,
        col_debut="heure_rlri16imm_debut_v4",  
        col_fin= "heure_nback_v4 (consignes)" ,         
        nom_bloc="bloc2"
    )




--- Feuille : APHM | bloc2 ---
Moyenne = 46.72 min
Écart-type = 9.14 min

Résumé rejeter_bloc2 :
rejeter_bloc2
0    18
2     7
Name: count, dtype: int64


--- Feuille : CAEN | bloc2 ---
🔧 Règle appliquée → centre=CAEN, bloc=bloc2, candidat=None
🔧 Règle appliquée → centre=CAEN, bloc=bloc2, candidat=0326BJR
Moyenne = 44.57 min
Écart-type = 6.41 min

Résumé rejeter_bloc2 :
rejeter_bloc2
0    48
1     1
2     7
Name: count, dtype: int64


--- Feuille : POITIERS | bloc2 ---
🔧 Règle appliquée → centre=POITIERS, bloc=bloc2, candidat=0408BCS
Moyenne = 46.00 min
Écart-type = 10.40 min

Résumé rejeter_bloc2 :
rejeter_bloc2
0    12
2     3
Name: count, dtype: int64


--- Feuille : ROUEN | bloc2 ---
Moyenne = 40.47 min
Écart-type = 5.58 min

Résumé rejeter_bloc2 :
rejeter_bloc2
0    15
Name: count, dtype: int64


--- Feuille : LAVERAN | bloc2 ---
Moyenne = 40.00 min
Écart-type = 13.14 min

Résumé rejeter_bloc2 :
rejeter_bloc2
0    4
2    1
Name: count, dtype: int64


--- Feuille : LPC | bloc2 --

## Bloc 3 

In [8]:
resultats_bloc3 = {}

for feuille in feuilles:
    print("\n" + "=" * 60)

    resultats_bloc3[feuille] = analyser_feuille_bloc(
        df=df[feuille],
        nom_feuille=feuille,
        col_debut="heure_nback_v4 (consignes)",   # à adapter si besoin
        col_fin="heure_fin_tests_v4",         # à adapter si besoin
        nom_bloc="bloc3"
    )



--- Feuille : APHM | bloc3 ---
Moyenne = 46.33 min
Écart-type = 13.57 min

Résumé rejeter_bloc3 :
rejeter_bloc3
0    18
2     7
Name: count, dtype: int64


--- Feuille : CAEN | bloc3 ---
🔧 Règle appliquée → centre=CAEN, bloc=bloc3, candidat=None
🔧 Règle appliquée → centre=CAEN, bloc=bloc3, candidat=0326BJR
Moyenne = 61.45 min
Écart-type = 198.00 min

Résumé rejeter_bloc3 :
rejeter_bloc3
0    49
2     7
Name: count, dtype: int64


--- Feuille : POITIERS | bloc3 ---
🔧 Règle appliquée → centre=POITIERS, bloc=bloc3, candidat=0408BCS
Moyenne = 43.50 min
Écart-type = 13.26 min

Résumé rejeter_bloc3 :
rejeter_bloc3
0    12
2     3
Name: count, dtype: int64


--- Feuille : ROUEN | bloc3 ---
Moyenne = 44.67 min
Écart-type = 6.21 min

Résumé rejeter_bloc3 :
rejeter_bloc3
0    14
1     1
Name: count, dtype: int64


--- Feuille : LAVERAN | bloc3 ---
Moyenne = 41.75 min
Écart-type = 9.03 min

Résumé rejeter_bloc3 :
rejeter_bloc3
0    4
2    1
Name: count, dtype: int64


--- Feuille : LPC | bloc3 

# Nombre de candidats qui passent toutes les conditions 

In [9]:
def candidats_valides_tous_blocs_depuis_resultats(
    feuille,
    resultats_blocs,
    col_id="Numero_inclusion"
):
    """
     Un candidat est conservé si nous pouvons calculer la durée de tous ses blocs et si celles-ci respectent les conditions suivantes :
    - bloc 1 > 10 min
    - bloc2R/S et bloc 3 > mean - 2σ min
    
    resultats_blocs = dict {
        "bloc1": resultats_bloc1,
        "bloc2": resultats_bloc2,
        "bloc3": resultats_bloc3
    }
    """

    # Récupération du df de référence dans lequel sont renseignés les durées des blocs
    df_ref = resultats_blocs["bloc1"][feuille]["df"].copy()

    colonnes_rejet = []

    # Ajouter chaque colonne de rejet depuis chaque bloc
    for nom_bloc, res_bloc in resultats_blocs.items():
        col_rejet = f"rejeter_{nom_bloc}"
        if col_rejet not in res_bloc[feuille]["df"].columns:
            print(f"Colonne manquante : {col_rejet} dans {feuille}")
            return None

        df_ref[col_rejet] = res_bloc[feuille]["df"][col_rejet]
        colonnes_rejet.append(col_rejet)

    # Condition : tout à 0
    mask_valide = (df_ref[colonnes_rejet] == 0).all(axis=1)

    candidats_ok = df_ref.loc[mask_valide, col_id].tolist()
    candidats_rejetes = df_ref.loc[~mask_valide, col_id].tolist()

    print(f"\n=== {feuille} ===")
    print(f"Candidats valides sur TOUS les blocs : {len(candidats_ok)}")

    print("Liste des candidats conservés :")
    for pid in candidats_ok:
        print(f" - {pid}")

    return {
        "nb_valides": len(candidats_ok),
        "valides": candidats_ok,
        "rejetes": candidats_rejetes,
    }


In [10]:
resultats_globaux = {}

colonnes_blocs = [
    "rejeter_bloc1",
    "rejeter_bloc2",
    "rejeter_bloc3"
]

for feuille in feuilles:
    print("\n" + "=" * 70)
    print(f"Analyse globale – Feuille : {feuille}")

    # ===== Base : bloc 1 =====
    df_global = resultats_bloc1[feuille]["df"][
        ["Numero_inclusion", "rejeter_bloc1"]
    ].copy()

    # ===== Fusion bloc 2 =====
    df_global = df_global.merge(
        resultats_bloc2[feuille]["df"][
            ["Numero_inclusion", "rejeter_bloc2"]
        ],
        on="Numero_inclusion",
        how="left"
    )

    # ===== Fusion bloc 3 =====
    df_global = df_global.merge(
        resultats_bloc3[feuille]["df"][
            ["Numero_inclusion", "rejeter_bloc3"]
        ],
        on="Numero_inclusion",
        how="left"
    )

    # ===== Affichage debug AVANT filtre =====
    print("\n📊 df_global AVANT fillna et filtres :")
    display(df_global)

    # ===== Sécurité : NaN → rejet (2) =====
    df_global[colonnes_blocs] = df_global[colonnes_blocs].fillna(2)

    mask = (
        (df_global["rejeter_bloc1"] == 0) &
        (df_global["rejeter_bloc2"] == 0) &
        (df_global["rejeter_bloc3"] == 0)
    )

    # ===== Extraction =====
    candidats_valides = df_global.loc[
        mask, "Numero_inclusion"
    ].tolist()

    candidats_rejetes = df_global.loc[
        ~mask, "Numero_inclusion"
    ].tolist()

    # ===== Résumés =====
    print(f"\nNombre total de candidats : {len(df_global)}")
    print(f"Candidats VALIDES (selon Condition) : {len(candidats_valides)}")
    print(f"Candidats REJETÉS : {len(candidats_rejetes)}")

    print("\nListe des candidats valides :")
    for pid in candidats_valides:
        print(f" - {pid}")

    # ===== Stockage =====
    resultats_globaux[feuille] = {
        "df": df_global,
        "valides": candidats_valides,
        "rejetes": candidats_rejetes,
        "n_valides": len(candidats_valides),
        "n_rejetes": len(candidats_rejetes),
    }



Analyse globale – Feuille : APHM

📊 df_global AVANT fillna et filtres :


,Numero_inclusion,rejeter_bloc1,rejeter_bloc2,rejeter_bloc3
0,0101CAR,0,0,0
1,0102PCR,0,0,0
2,0103SHS,1,0,0
3,0105PNR,0,0,0
4,0104FJS,0,0,0
5,0106JLS,1,0,0
6,0107DSS,0,0,0
7,0108BFS,1,0,0
8,0109GSS,0,0,0
9,0110LPR,0,0,0



Nombre total de candidats : 25
Candidats VALIDES (selon Condition) : 15
Candidats REJETÉS : 10

Liste des candidats valides :
 - 0101CAR
 - 0102PCR
 - 0105PNR
 - 0104FJS
 - 0107DSS
 - 0109GSS
 - 0110LPR
 - 0111MNR
 - 0112BSR
 - 0114LMS
 - 0116VAR
 - 0119LJS
 - 0121MJR
 - 0122BDS
 - 0123RMS

Analyse globale – Feuille : CAEN

📊 df_global AVANT fillna et filtres :


,Numero_inclusion,rejeter_bloc1,rejeter_bloc2,rejeter_bloc3
0,0301GNR,2,2,2
1,0302ZMR,0,0,0
2,0303PAR,2,2,2
3,0304VMR,0,0,0
4,0305LCR,0,0,0
5,0307SMR,0,0,0
6,0306MDR,1,0,0
7,0308RGR,0,0,0
8,0309DBS,2,2,2
9,0311CJS,0,0,0



Nombre total de candidats : 56
Candidats VALIDES (selon Condition) : 38
Candidats REJETÉS : 18

Liste des candidats valides :
 - 0302ZMR
 - 0304VMR
 - 0305LCR
 - 0307SMR
 - 0308RGR
 - 0311CJS
 - 0310ACS
 - 0315VCS
 - 0313FPR
 - 0316TMS
 - 0317LGS
 - 0319LJS
 - 0321DRR
 - 0323RFS
 - 0322RMS
 - 0324LMR
 - 0326BJR
 - 0328MPR
 - 0330RTR
 - 0329NMR
 - 0331GRS
 - 0333MMS
 - 0337BFS
 - 0338JBS
 - 0339NPR
 - 0340BAR
 - 0341LJS
 - 0342VNS
 - 0343PAS
 - 0345LAR
 - 0349LLS
 - 0347DMR
 - 0346GLS
 - 0348GCR
 - 0350MYR
 - 0351FIR
 - 0352RNR
 - 0356DMS

Analyse globale – Feuille : POITIERS

📊 df_global AVANT fillna et filtres :


...


Nombre total de candidats : 8
Candidats VALIDES (selon Condition) : 7
Candidats REJETÉS : 1

Liste des candidats valides :
 - 0801HDR
 - 0802LAS
 - 0804GOR
 - 0803DPS
 - 0805BMS
 - 0806KHS
 - 0807OMR

Analyse globale – Feuille : CGD

📊 df_global AVANT fillna et filtres :


,Numero_inclusion,rejeter_bloc1,rejeter_bloc2,rejeter_bloc3
0,0901SMR,0,0,0
1,0903DJS,0,0,0
2,0902SIS,0,0,0
3,0906DNR,2,2,2
4,0904KJS,0,0,0
5,0905SLR,0,0,0



Nombre total de candidats : 6
Candidats VALIDES (selon Condition) : 5
Candidats REJETÉS : 1

Liste des candidats valides :
 - 0901SMR
 - 0903DJS
 - 0902SIS
 - 0904KJS
 - 0905SLR

Analyse globale – Feuille : CERCA

📊 df_global AVANT fillna et filtres :


,Numero_inclusion,rejeter_bloc1,rejeter_bloc2,rejeter_bloc3
0,0401BDS,0,0,0
1,0402TFR,2,2,2
2,0404DER,0,0,0
3,0405RCR,0,0,0
4,0406BBS,0,0,0
5,0407HMS,2,2,2
6,0408SJS,0,0,0
7,0409HCR,0,0,0
8,0410FMS,0,0,0
9,0411NPR,2,2,2



Nombre total de candidats : 40
Candidats VALIDES (selon Condition) : 11
Candidats REJETÉS : 29

Liste des candidats valides :
 - 0401BDS
 - 0404DER
 - 0405RCR
 - 0406BBS
 - 0408SJS
 - 0409HCR
 - 0410FMS
 - 0414PVR
 - 0418MPR
 - 0420MCR
 - 0423SVS


In [11]:
total_valides = sum(
    res["n_valides"] for res in resultats_globaux.values()
)

print("=" * 70)
print(f"✅ Nombre TOTAL de candidats valides (toutes feuilles confondues) : {total_valides}")

✅ Nombre TOTAL de candidats valides (toutes feuilles confondues) : 143


# 144 candidats retenus

# Recherche des candidats et téléchargement de leur fichier V4 BB RR

In [12]:
import pandas as pd

# Afficher toutes les lignes
pd.set_option('display.max_rows', None)

# Afficher toutes les colonnes
pd.set_option('display.max_columns', None)

# Afficher toute la largeur de chaque colonne
pd.set_option('display.max_colwidth', None)

## APHM

In [13]:
# ===== Racine aphm =====
racine_aphm = r"C:\Users\judupont\Desktop\AGING_19_01_2026\LPC\GAUTHIER\GAUTHIER\PATIENTS"

# ===== Exemple pour un centre (à dupliquer pour chaque centre) =====
centre = "APHM"
candidats_aphm = [c.upper() for c in resultats_globaux[centre]["valides"]]

rows = []

for root, dirs, files in os.walk(racine_aphm):

    if "V4" not in root.upper():
        continue

    for f in files:

        # Seulement CSV
        if not f.lower().endswith(".csv"):
            continue
        # Seulement BB_RR ou BR_RR
        if not re.search(r"(BB_RR|BR_RR)", f, re.IGNORECASE):
            continue

        chemin = os.path.join(root, f)
        chemin_upper = chemin.upper()

        # ===== Détection du Numero_inclusion dans le chemin =====
        pid_trouve = None
        for pid in candidats_aphm:
            if pid in chemin_upper:
                pid_trouve = pid
                break
        if not pid_trouve:
            continue

        rows.append({
            "Numero_inclusion": pid_trouve,
            "Chemin": chemin,
            "Feuille": centre
        })

# ===== DataFrame =====
df_aphm = pd.DataFrame(rows)

# ===== Suppression doublons =====
df_aphm_final = (
    df_aphm
    .assign(priorite_ceinture=df_aphm["Chemin"].str.contains("CEINTURE", case=False))
    .sort_values(["Numero_inclusion", "priorite_ceinture"], ascending=[True, False])
    .drop_duplicates(subset="Numero_inclusion", keep="first")
    .drop(columns="priorite_ceinture")
    .reset_index(drop=True)
)

print(f"Candidats attendus {centre} : {len(candidats_aphm)}")
print(f"Fichiers BB/BR_RR {centre} V4 retenus : {len(df_aphm_final)}")

display(df_aphm_final)

Candidats attendus APHM : 15
Fichiers BB/BR_RR APHM V4 retenus : 15


,Numero_inclusion,Chemin,Feuille
0,0101CAR,C:\Users\judupont\Desktop\AGING_19_01_2026\LPC\GAUTHIER\GAUTHIER\PATIENTS\TOUS_patients_APHM_laveran\0101CAR\0101CAR 04-04-2019 V4\Ceinture 0101CAR\2019_04_04-08_34_38_BB_RR.csv,APHM
1,0102PCR,C:\Users\judupont\Desktop\AGING_19_01_2026\LPC\GAUTHIER\GAUTHIER\PATIENTS\TOUS_patients_APHM_laveran\0102PCR\0102PCR 12-04-2019 V4\Ceinture 0102PCR\2019_04_12-09_06_05_BB_RR.csv,APHM
2,0104FJS,C:\Users\judupont\Desktop\AGING_19_01_2026\LPC\GAUTHIER\GAUTHIER\PATIENTS\TOUS_patients_APHM_laveran\0104FJS\0104FJS 08-11-2019 V4\Ceinture 0104FJS\2019_11_08-08_26_41_BB_RR.csv,APHM
3,0105PNR,C:\Users\judupont\Desktop\AGING_19_01_2026\LPC\GAUTHIER\GAUTHIER\PATIENTS\TOUS_patients_APHM_laveran\0105PNR\0105PNR 09-09-2019 V4\Ceinture 0105PNR\2019_09_09-08_53_04_BB_RR.csv,APHM
4,0107DSS,C:\Users\judupont\Desktop\AGING_19_01_2026\LPC\GAUTHIER\GAUTHIER\PATIENTS\TOUS_patients_APHM_laveran\0107DSS\0107DSS 17-10-2019 V4\Ceinture 0107DSS\2019_10_17-09_08_45_BB_RR.csv,APHM
5,0109GSS,C:\Users\judupont\Desktop\AGING_19_01_2026\LPC\GAUTHIER\GAUTHIER\PATIENTS\TOUS_patients_APHM_laveran\0109GSS\0109GSS 14-11-2019 V4\Ceinture 0109GSS\2019_11_14-09_36_14_BB_RR.csv,APHM
6,0110LPR,C:\Users\judupont\Desktop\AGING_19_01_2026\LPC\GAUTHIER\GAUTHIER\PATIENTS\TOUS_patients_APHM_laveran\0110LPR\0110LPR 21-11-2019 V4\Ceinture 0110LPR\2019_11_21-09_04_12_BB_RR.csv,APHM
7,0111MNR,C:\Users\judupont\Desktop\AGING_19_01_2026\LPC\GAUTHIER\GAUTHIER\PATIENTS\TOUS_patients_APHM_laveran\0111MNR\0111MNR 22-11-2019 V4\Ceinture 0111MNR\2019_11_22-09_17_01_BB_RR.csv,APHM
8,0112BSR,C:\Users\judupont\Desktop\AGING_19_01_2026\LPC\GAUTHIER\GAUTHIER\PATIENTS\TOUS_patients_APHM_laveran\0112BSR\0112BSR - V4\V4\ceinture\2020_03_05-09_38_29\2020_03_05-09_38_29_BB_RR.csv,APHM
9,0114LMS,C:\Users\judupont\Desktop\AGING_19_01_2026\LPC\GAUTHIER\GAUTHIER\PATIENTS\TOUS_patients_APHM_laveran\0114LMS\0114LMS_V4\ceinture\2020_09_04-09_43_24\2020_09_04-09_43_24_BB_RR.csv,APHM


## CAEN

In [15]:
# ======================================================
# RACINE CEINTURE CAEN
# ======================================================
racine_caen = r"C:\Users\judupont\Desktop\AGING_19_01_2026\CHU-CAEN\ALLAIN\ALLAIN\CEINTURE"

# ======================================================
# EXTRACTION DATE_V4 ET FILTRE CANDIDATS VALIDES
# ======================================================
df_caen_excel = df["CAEN"].copy()  # contient la colonne Date_v4

# Normalisation des colonnes
df_caen_excel.columns = (
    df_caen_excel.columns
    .str.strip()
    .str.lower()
    .str.replace(" ", "_")
)

COL_PID = "numero_inclusion"
COL_DATE = "date_v4"

# Filtrer seulement les candidats valides
candidats_valide = [c.upper() for c in resultats_globaux["CAEN"]["valides"]]
df_caen_excel = df_caen_excel[[COL_PID, COL_DATE]].dropna()
df_caen_excel[COL_PID] = df_caen_excel[COL_PID].str.upper()
df_caen_excel = df_caen_excel[df_caen_excel[COL_PID].isin(candidats_valide)]

# Conversion date Excel → datetime
df_caen_excel[COL_DATE] = pd.to_datetime(df_caen_excel[COL_DATE], errors="coerce", dayfirst=True)
df_caen_excel = df_caen_excel.dropna(subset=[COL_DATE])

# Création string YYYY_MM_DD pour matcher avec le nom du fichier
df_caen_excel["date_str"] = df_caen_excel[COL_DATE].dt.strftime("%Y_%m_%d")
dict_date_v4 = dict(zip(df_caen_excel[COL_PID], df_caen_excel["date_str"]))

# ======================================================
# PARCOURS DES FICHIERS CEINTURE CAEN
# ======================================================
rows = []

for root, dirs, files in os.walk(racine_caen):
    for f in files:

        if not f.lower().endswith(".csv"):
            continue
        if not re.search(r"BB_RR|BR_RR", f, re.IGNORECASE):
            continue

        chemin = os.path.join(root, f)
        chemin_upper = chemin.upper()

        # ===== PID =====
        pid_trouve = None
        for pid in dict_date_v4:
            if pid in chemin_upper:
                pid_trouve = pid
                break
        if not pid_trouve:
            continue

        # ===== DATE fichier =====
        match_date = re.search(r"\d{4}[_-]\d{2}[_-]\d{2}", f)
        if not match_date:
            continue
        date_fichier = match_date.group().replace("-", "_")

        # ===== Comparaison avec la date V4 Excel =====
        if date_fichier != dict_date_v4[pid_trouve]:
            # Ignorer normalement si date différente
            # Mais pour certains cas on forcera plus bas
            continue

        rows.append({
            "Numero_inclusion": pid_trouve,
            "Chemin": chemin,
            "Feuille": "CAEN"
        })

# ======================================================
# FORCER LES PATHS POUR LES CAS SPÉCIAUX
# ======================================================
#  Ces candidats sont échangés dans la base de donnée, donc on force les chemins des fichiers BB_RR
forced_paths = {
    "0342VNS": r"C:\Users\judupont\Desktop\AGING_19_01_2026_copie\CHU-CAEN\ALLAIN\ALLAIN\CEINTURE\2022_03_08_10_03_11 0343PAS\2022_03_08__10_03_11_BR_RR.csv",
    "0343PAS": r"C:\Users\judupont\Desktop\AGING_19_01_2026_copie\CHU-CAEN\ALLAIN\ALLAIN\CEINTURE\2022_04_01_09_38_38 0342VNS\2022_04_01__09_38_38_BR_RR.csv"
}

for pid, path_forced in forced_paths.items():
    rows.append({
        "Numero_inclusion": pid,
        "Chemin": path_forced,
        "Feuille": "CAEN"
    })

# ======================================================
# DATAFRAME FINAL CAEN V4
# ======================================================
df_caen_final = pd.DataFrame(rows).sort_values("Numero_inclusion").reset_index(drop=True)

# ======================================================
# SUPPRESSION DES DOUBLONS (on garde la première occurrence du Numero_inclusion)
# ======================================================
df_caen_final = df_caen_final.drop_duplicates(subset="Numero_inclusion", keep="first").reset_index(drop=True)

print(f"Candidats CAEN V4 attendus : {len(candidats_valide)}")
print(f"Candidats CAEN V4 retenus après suppression des doublons : {len(df_caen_final)}")

display(df_caen_final)

Candidats CAEN V4 attendus : 38
Candidats CAEN V4 retenus après suppression des doublons : 38


,Numero_inclusion,Chemin,Feuille
0,0302ZMR,C:\Users\judupont\Desktop\AGING_19_01_2026\CHU-CAEN\ALLAIN\ALLAIN\CEINTURE\2019_10_22-10_08_57 0302ZMR\2019_10_22-10_08_57_BB_RR.csv,CAEN
1,0304VMR,C:\Users\judupont\Desktop\AGING_19_01_2026\CHU-CAEN\ALLAIN\ALLAIN\CEINTURE\2020_01_24-09_55_49 0304VMR\2020_01_24-09_55_49_BB_RR.csv,CAEN
2,0305LCR,C:\Users\judupont\Desktop\AGING_19_01_2026\CHU-CAEN\ALLAIN\ALLAIN\CEINTURE\2020_01_23-09_51_21 0305LCR\2020_01_23-09_51_21_BB_RR (1).csv,CAEN
3,0307SMR,C:\Users\judupont\Desktop\AGING_19_01_2026\CHU-CAEN\ALLAIN\ALLAIN\CEINTURE\2020_05_25-10_35_35 0307SMR\2020_05_25-10_35_35_BB_RR.csv,CAEN
4,0308RGR,C:\Users\judupont\Desktop\AGING_19_01_2026\CHU-CAEN\ALLAIN\ALLAIN\CEINTURE\2020_09_02_09_04_56 0308RGR\2020_09_02__09_04_56_BR_RR (1).csv,CAEN
5,0310ACS,C:\Users\judupont\Desktop\AGING_19_01_2026\CHU-CAEN\ALLAIN\ALLAIN\CEINTURE\2020_05_18-08_40_01 0310ACS\2020_05_18-08_40_01_BB_RR (1).csv,CAEN
6,0311CJS,C:\Users\judupont\Desktop\AGING_19_01_2026\CHU-CAEN\ALLAIN\ALLAIN\CEINTURE\2020_06_17-09_10_51 0311CJS\2020_06_17__09_10_51_BR_RR.csv,CAEN
7,0313FPR,C:\Users\judupont\Desktop\AGING_19_01_2026\CHU-CAEN\ALLAIN\ALLAIN\CEINTURE\2020_06_08-10_01_39 0313FPR\2020_06_08-10_01_39_BB_RR.csv,CAEN
8,0315VCS,C:\Users\judupont\Desktop\AGING_19_01_2026\CHU-CAEN\ALLAIN\ALLAIN\CEINTURE\2020_06_11-09_44_57 0315VCS\2020_06_11-09_44_57_BB_RR.csv,CAEN
9,0316TMS,C:\Users\judupont\Desktop\AGING_19_01_2026\CHU-CAEN\ALLAIN\ALLAIN\CEINTURE\2020_06_02-09_58_49 0316TMS\2020_06_02-09_58_49_BB_RR.csv,CAEN


## POITIERS

In [17]:
# ===== Racine POITIERS =====
racine_poitiers = r"C:\Users\judupont\Desktop\AGING_19_01_2026\CHU-POITIERS\MARTIN DE MONTAUDRY"

# ===== Candidats valides =====
candidats_poitiers = [c.upper() for c in resultats_globaux["POITIERS"]["valides"]]

# ===== Regex POITIERS (tronquée → clé pour 0408 etc) =====
patterns_candidats = {
    pid: re.compile(rf"{pid[:4]}\s*-?\s*{pid[4:6]}", re.IGNORECASE)
    for pid in candidats_poitiers
}

# ===== Collecte brute des fichiers =====
fichiers_par_candidat = defaultdict(list)

for root, dirs, files in os.walk(racine_poitiers):
    if "V4" not in root.upper():
        continue

    for f in files:
        if not f.lower().endswith(".csv"):
            continue
        if not re.search(r"(BB_RR|BR_RR)", f, re.IGNORECASE):
            continue

        chemin = os.path.join(root, f)

        for pid, pat in patterns_candidats.items():
            if pat.search(chemin):
                fichiers_par_candidat[pid].append(chemin)
                break

# ===== Sélection finale + DataFrame =====
rows = []

for pid, fichiers in fichiers_par_candidat.items():
    fichiers = sorted(set(fichiers))  # enlever doublons exacts

    # priorité CEINTURE
    ceinture = [f for f in fichiers if "ceinture" in f.lower()]
    chemin_final = ceinture[0] if ceinture else fichiers[0]

    rows.append({
        "Numero_inclusion": pid,
        "Chemin": chemin_final,
        "Feuille": "POITIERS"
    })

df_poitiers_final = pd.DataFrame(rows)

# ===== Vérifications =====
print(f"Candidats attendus POITIERS : {len(candidats_poitiers)}")
print(f"Fichiers BB/BR_RR POITIERS V4 retenus : {len(df_poitiers_final)}")

manquants = sorted(set(candidats_poitiers) - set(df_poitiers_final["Numero_inclusion"]))
if manquants:
    print("⚠️ Candidats manquants :", manquants)
else:
    print("✅ Tous les candidats POITIERS V4 sont présents")

display(df_poitiers_final)

Candidats attendus POITIERS : 11
Fichiers BB/BR_RR POITIERS V4 retenus : 10
⚠️ Candidats manquants : ['0415VMR']


,Numero_inclusion,Chemin,Feuille
0,0401TSS,C:\Users\judupont\Desktop\AGING_19_01_2026\CHU-POITIERS\MARTIN DE MONTAUDRY\MARTIN DE MONTAUDRY\0401 TS\0401TSS - V4\ceinture\No Team Assigned\2020_01_10-09_57_29\2020_01_10-09_57_29_BB_RR.csv,POITIERS
1,0402LLS,C:\Users\judupont\Desktop\AGING_19_01_2026\CHU-POITIERS\MARTIN DE MONTAUDRY\MARTIN DE MONTAUDRY\0402 LL\0402LLS-V4\0402LLS ceinture V4\No Team Assigned\2020_02_07-09_30_21\2020_02_07-09_30_21_BB_RR.csv,POITIERS
2,0403DCR,C:\Users\judupont\Desktop\AGING_19_01_2026\CHU-POITIERS\MARTIN DE MONTAUDRY\MARTIN DE MONTAUDRY\0403 DC\0403DCR-V4\0403DCR - ceinture V4\No Team Assigned\2020_02_28-10_11_17\2020_02_28-10_11_17_BB_RR.csv,POITIERS
3,0404CYS,C:\Users\judupont\Desktop\AGING_19_01_2026\CHU-POITIERS\MARTIN DE MONTAUDRY\MARTIN DE MONTAUDRY\0404 CY\0404YCS-V4\0404YC ceinture V4\2020_06_19-10_14_18_BB_RR.csv,POITIERS
4,0407LJR,C:\Users\judupont\Desktop\AGING_19_01_2026\CHU-POITIERS\MARTIN DE MONTAUDRY\MARTIN DE MONTAUDRY\0407 LJ\0407LJR-V4\0407LJ ceinture V4\No Team Assigned\2020_07_02-09_52_07\2020_07_02-09_52_07_BB_RR.csv,POITIERS
5,0408BCS,C:\Users\judupont\Desktop\AGING_19_01_2026\CHU-POITIERS\MARTIN DE MONTAUDRY\MARTIN DE MONTAUDRY\0408 BC\0408 BC-V4\0408BC-ceinture V4\No Team Assigned\2020_08_06-10_25_47\2020_08_06-10_25_47_BB_RR.csv,POITIERS
6,0411VES,C:\Users\judupont\Desktop\AGING_19_01_2026\CHU-POITIERS\MARTIN DE MONTAUDRY\MARTIN DE MONTAUDRY\0411 VE\0411 VES-V4\0411VE-ceinture V4\0411VES - V4\2020_11_12-10_17_49\2020_11_12-10_17_49_BB_RR.csv,POITIERS
7,0412ANS,C:\Users\judupont\Desktop\AGING_19_01_2026\CHU-POITIERS\MARTIN DE MONTAUDRY\MARTIN DE MONTAUDRY\0412 AN\0412 ANS-V4\0412ANS ceinture V4\No Team Assigned\2020_11_19-10_02_29\2020_11_19-10_02_29_BB_RR.csv,POITIERS
8,0413HFS,C:\Users\judupont\Desktop\AGING_19_01_2026\CHU-POITIERS\MARTIN DE MONTAUDRY\MARTIN DE MONTAUDRY\0413 HF\0413 HFS V4\0413HFS-ceinture V4\No Team Assigned\2020_11_26-09_37_17\2020_11_26-09_37_17_BB_RR.csv,POITIERS
9,0414PJS,C:\Users\judupont\Desktop\AGING_19_01_2026\CHU-POITIERS\MARTIN DE MONTAUDRY\MARTIN DE MONTAUDRY\0414 PJ\0414 PJ V4\0414PJ-ceinture V4\2021_06_10-09_47_37\2021_06_10-09_47_37_BB_RR.csv,POITIERS


## ROUEN

In [19]:
# ===== Racine ROUEN =====
racine_rouen = r"C:\Users\judupont\Desktop\AGING_19_01_2026\CHU-ROUEN\HANNIER"

# ===== Candidats valides =====
candidats_rouen = [c.upper() for c in resultats_globaux["ROUEN"]["valides"]]

# ===== paramètres poids =====
POIDS_MIN = 3000 * 1024
POIDS_MAX = 8000 * 1024

# ===== pattern numéro (ex : 05-12) =====
patterns_candidats = [
    (pid, re.compile(rf"{pid[:2]}-{pid[2:4]}", re.IGNORECASE))
    for pid in candidats_rouen
]

# ===== pattern date dans le fichier =====
date_pattern = re.compile(r"(\d{4}_\d{2}_\d{2}[-_]\d{2}_\d{2}_\d{2})")

fichiers_dict = defaultdict(list)

for root, dirs, files in os.walk(racine_rouen):

    if "V4" not in root.upper():
        continue

    for f in files:

        if not f.lower().endswith(".csv"):
            continue

        if not re.search(r"(BB_RR|BR_RR)", f, re.IGNORECASE):
            continue

        chemin = os.path.join(root, f)

        # ---- filtre poids ----
        try:
            taille = os.path.getsize(chemin)
        except OSError:
            continue

        if not (POIDS_MIN <= taille <= POIDS_MAX):
            continue

        # ---- identification candidat ----
        pid_trouve = None
        for pid, pat in patterns_candidats:
            if pat.search(chemin):
                pid_trouve = pid
                break

        if not pid_trouve:
            continue

        # ---- date ----
        m = date_pattern.search(chemin)
        date_str = m.group(1) if m else "inconnue"

        fichiers_dict[(pid_trouve, date_str)].append(chemin)

# ===== Sélection finale + DataFrame =====
rows = []

for (pid, date_str), fichiers in fichiers_dict.items():

    fichiers = sorted(set(fichiers)) 

    ceinture = [f for f in fichiers if "ceinture" in f.lower()]
    chemin_final = ceinture[0] if ceinture else fichiers[0]

    rows.append({
        "Numero_inclusion": pid,
        "Chemin": chemin_final,
        "Feuille": "ROUEN"
    })

df_rouen_final = pd.DataFrame(rows)

# ===== Résumé & contrôle =====
print(f"Candidats attendus ROUEN : {len(candidats_rouen)}")
print(f"Fichiers BB/BR_RR ROUEN V4 retenus : {len(df_rouen_final)}")

manquants = sorted(set(candidats_rouen) - set(df_rouen_final["Numero_inclusion"]))
if manquants:
    print("⚠️ Candidats ROUEN manquants :", manquants)
else:
    print("✅ Tous les candidats ROUEN V4 sont présents")

display(df_rouen_final)

# On perd le 0509LGS car pas de V4 dans l'arbo

Candidats attendus ROUEN : 13
Fichiers BB/BR_RR ROUEN V4 retenus : 12
⚠️ Candidats ROUEN manquants : ['0509LGS']


,Numero_inclusion,Chemin,Feuille
0,0501MBS,C:\Users\judupont\Desktop\AGING_19_01_2026\CHU-ROUEN\HANNIER\HANNIER\05-01-MB-S\V4\CEINTURE\2019_11_18-09_10_13_BB_RR - Copie.csv,ROUEN
1,0502LIR,C:\Users\judupont\Desktop\AGING_19_01_2026\CHU-ROUEN\HANNIER\HANNIER\05-02-LI-R\V4\CEINTURE\2019_11_13-08_56_31_BB_RR.csv,ROUEN
2,0503LCR,C:\Users\judupont\Desktop\AGING_19_01_2026\CHU-ROUEN\HANNIER\HANNIER\05-03-LC-R\V4\CEINTURE\2020_01_23-09_24_20_BB_RR.csv,ROUEN
3,0504BCS,C:\Users\judupont\Desktop\AGING_19_01_2026\CHU-ROUEN\HANNIER\HANNIER\05-04-BC-S\V4\CEINTURE V4\2020_03_05-09_10_25_BB_RR.csv,ROUEN
4,0505PPR,C:\Users\judupont\Desktop\AGING_19_01_2026\CHU-ROUEN\HANNIER\HANNIER\05-05-PP-R\V4\CEINTURE\2020_02_14-09_12_56_BB_RR.csv,ROUEN
5,0507BDS,C:\Users\judupont\Desktop\AGING_19_01_2026\CHU-ROUEN\HANNIER\HANNIER\05-07-BD-S\V4\CEINTURE\2020_09_24-09_15_56_BB_RR.csv,ROUEN
6,0508SRR,C:\Users\judupont\Desktop\AGING_19_01_2026\CHU-ROUEN\HANNIER\HANNIER\05-08-SR-R\V4\CEINTURE\2020_05_12-09_15_51_BB_RR.csv,ROUEN
7,0510DFS,C:\Users\judupont\Desktop\AGING_19_01_2026\CHU-ROUEN\HANNIER\HANNIER\05-10-DF-S\V4\CEINTURE\2020_11_17-09_13_35_BB_RR.csv,ROUEN
8,0512RCR,C:\Users\judupont\Desktop\AGING_19_01_2026\CHU-ROUEN\HANNIER\HANNIER\05-12-RC-R\V4\CEINTURE\Record 1_2022_02_15-14_05_19_BB_RR.csv,ROUEN
9,0513EBS,C:\Users\judupont\Desktop\AGING_19_01_2026\CHU-ROUEN\HANNIER\HANNIER\05-13-EB-S\V4\CEINTURE\Record 1_2022_03_15-09_12_51_BB_RR - Copie.csv,ROUEN


## LAVERAN

In [21]:
# ===== Racine LAVERAN =====
racine_laveran = (
    r"C:\Users\judupont\Desktop\AGING_19_01_2026"
    r"\LPC\GAUTHIER\GAUTHIER\PATIENTS\TOUS_patients_APHM_laveran"
)

# ===== Candidats attendus =====
candidats_laveran = [c.upper() for c in resultats_globaux["LAVERAN"]["valides"]]

rows = []

# =========================================================
# BOUCLE PRINCIPALE
# =========================================================
for root, dirs, files in os.walk(racine_laveran):

    if "V4" not in root.upper():
        continue

    for f in files:

        # CSV BB/BR_RR uniquement
        if not f.lower().endswith(".csv"):
            continue
        if not re.search(r"(BB_RR|BR_RR)", f, re.IGNORECASE):
            continue

        chemin = os.path.join(root, f)
        chemin_upper = chemin.upper()

        # ===== identification du candidat =====
        pid = None
        for c in candidats_laveran:
            if c in chemin_upper:
                pid = c
                break

        if not pid:
            continue

        rows.append({
            "Numero_inclusion": pid,
            "Chemin": chemin,
            "Feuille": "LAVERAN"
        })

        print(f"OK [{pid}] :", chemin)

# =========================================================
# DATAFRAME + SUPPRESSION DES DOUBLONS
# =========================================================
df_laveran_final = pd.DataFrame(rows)

# 1 fichier max par candidat
df_laveran_final = (
    df_laveran_final
    .drop_duplicates(subset="Numero_inclusion", keep="first")
    .reset_index(drop=True)
)

# =========================================================
# RÉSUMÉ & CONTRÔLES
# =========================================================
print("\n====================================")
print(f"Candidats attendus LAVERAN : {len(candidats_laveran)}")
print(f"Fichiers retenus LAVERAN    : {len(df_laveran_final)}")

manquants = sorted(
    set(candidats_laveran) - set(df_laveran_final["Numero_inclusion"])
)

if manquants:
    print("⚠️ Candidats LAVERAN manquants :", manquants)
else:
    print("✅ Tous les candidats LAVERAN sont présents")

display(df_laveran_final)


OK [0626MCS] : C:\Users\judupont\Desktop\AGING_19_01_2026\LPC\GAUTHIER\GAUTHIER\PATIENTS\TOUS_patients_APHM_laveran\0626MCS\V4\0626MCS_v4\CEINTURE\No Team Assigned\2020_10_29-10_17_45\2020_10_29-10_17_45_BB_RR.csv
OK [0629TCR] : C:\Users\judupont\Desktop\AGING_19_01_2026\LPC\GAUTHIER\GAUTHIER\PATIENTS\TOUS_patients_APHM_laveran\0629TCR\0629TCR_V4\CEINTURE\No Team Assigned\2020_11_17-09_49_55\2020_11_17-09_49_55_BB_RR.csv
OK [0630RJR] : C:\Users\judupont\Desktop\AGING_19_01_2026\LPC\GAUTHIER\GAUTHIER\PATIENTS\TOUS_patients_APHM_laveran\0630RJR\0630RJR_V4\CEINTURE\2020_11_16-10_07_29\2020_11_16-10_07_29_BB_RR.csv
OK [0631LHR] : C:\Users\judupont\Desktop\AGING_19_01_2026\LPC\GAUTHIER\GAUTHIER\PATIENTS\TOUS_patients_APHM_laveran\0631LHR\0631LHR_V4\CEINTURE\2020_11_18-09_33_29\2020_11_18-09_33_29_BB_RR.csv

Candidats attendus LAVERAN : 4
Fichiers retenus LAVERAN    : 4
✅ Tous les candidats LAVERAN sont présents


,Numero_inclusion,Chemin,Feuille
0,0626MCS,C:\Users\judupont\Desktop\AGING_19_01_2026\LPC\GAUTHIER\GAUTHIER\PATIENTS\TOUS_patients_APHM_laveran\0626MCS\V4\0626MCS_v4\CEINTURE\No Team Assigned\2020_10_29-10_17_45\2020_10_29-10_17_45_BB_RR.csv,LAVERAN
1,0629TCR,C:\Users\judupont\Desktop\AGING_19_01_2026\LPC\GAUTHIER\GAUTHIER\PATIENTS\TOUS_patients_APHM_laveran\0629TCR\0629TCR_V4\CEINTURE\No Team Assigned\2020_11_17-09_49_55\2020_11_17-09_49_55_BB_RR.csv,LAVERAN
2,0630RJR,C:\Users\judupont\Desktop\AGING_19_01_2026\LPC\GAUTHIER\GAUTHIER\PATIENTS\TOUS_patients_APHM_laveran\0630RJR\0630RJR_V4\CEINTURE\2020_11_16-10_07_29\2020_11_16-10_07_29_BB_RR.csv,LAVERAN
3,0631LHR,C:\Users\judupont\Desktop\AGING_19_01_2026\LPC\GAUTHIER\GAUTHIER\PATIENTS\TOUS_patients_APHM_laveran\0631LHR\0631LHR_V4\CEINTURE\2020_11_18-09_33_29\2020_11_18-09_33_29_BB_RR.csv,LAVERAN


## LPC

In [22]:
# ===== Racine LPC =====
racine_lpc = r"C:\Users\judupont\Desktop\AGING_19_01_2026\LNSC\DESHAYES\DESHAYES\DATA_Aging"

# ===== Candidats attendus =====
candidats_lpc = [c.upper() for c in resultats_globaux["LPC"]["valides"]]

# ===== paramètres poids =====
POIDS_MIN = 2000 * 1024   # 2 Mo
POIDS_MAX = 8000 * 1024   # 8 Mo

# ===== pattern date dans le nom de fichier =====
date_pattern = re.compile(r"(\d{4}_\d{2}_\d{2}-\d{2}_\d{2}_\d{2})")

rows = []

for root, dirs, files in os.walk(racine_lpc):

    if "V4" not in root.upper():
        continue

    for f in files:

        # CSV BB/BR_RR uniquement
        if not f.lower().endswith(".csv"):
            continue
        if not re.search(r"(BB_RR|BR_RR)", f, re.IGNORECASE):
            continue

        chemin = os.path.join(root, f)
        chemin_upper = chemin.upper()

        # ---- filtre poids ----
        try:
            taille = os.path.getsize(chemin)
        except OSError:
            continue
        if not (POIDS_MIN <= taille <= POIDS_MAX):
            continue

        # ---- identification candidat ----
        pid = None
        for c in candidats_lpc:
            if c == "0117MSR" and "0117MMR" in chemin_upper:
                pid = c  # correction manuelle
                break
            if c in chemin_upper:
                pid = c
                break
        if not pid:
            continue

        # ---- date dans le fichier ----
        m = date_pattern.search(f)
        date_str = m.group(1) if m else "inconnue"

        rows.append({
            "Numero_inclusion": pid,
            "Chemin": chemin,
            "Feuille": "LPC"
        })

# ===== SUPPRESSION DES DOUBLONS =====
df_lpc_final = pd.DataFrame(rows)
df_lpc_final = (
    df_lpc_final
    .sort_values(["Numero_inclusion"])
    .drop_duplicates(subset="Numero_inclusion", keep="first")
    .reset_index(drop=True)
)

# ===== AFFICHAGE =====
print(f"Candidats attendus LPC : {len(candidats_lpc)}")
print(f"Fichiers BB/BR_RR V4 retenus : {len(df_lpc_final)}")

manquants = sorted(set(candidats_lpc) - set(df_lpc_final["Numero_inclusion"]))
if manquants:
    print("⚠️ Candidats LPC manquants :", manquants)
else:
    print("✅ Tous les candidats LPC sont présents")

display(df_lpc_final)

Candidats attendus LPC : 39
Fichiers BB/BR_RR V4 retenus : 26
⚠️ Candidats LPC manquants : ['0106DJR', '0130FAR', '0132GAR', '0133BPS', '0136CJR', '0137BMS', '0138NWR', '0139BMR', '0140LMR', '0141HJR', '0142LLS', '0143EBR', '0144ZGR']


,Numero_inclusion,Chemin,Feuille
0,0101EMS,C:\Users\judupont\Desktop\AGING_19_01_2026\LNSC\DESHAYES\DESHAYES\DATA_Aging\0101EMS\0101EMS V4\Ceinture\2019_10_22-09_42_52\2019_10_22-09_42_52_BB_RR.csv,LPC
1,0102EMS,C:\Users\judupont\Desktop\AGING_19_01_2026\LNSC\DESHAYES\DESHAYES\DATA_Aging\0102EMS\V4 0102EMS\Ceinture\2019_09_05-09_36_48\2019_09_05-09_36_48_BB_RR.csv,LPC
2,0103BPS,C:\Users\judupont\Desktop\AGING_19_01_2026\LNSC\DESHAYES\DESHAYES\DATA_Aging\0103BPS\V4 0103BPS\Ceinture\2019_09_17-09_43_26\2019_09_17-09_43_26_BB_RR.csv,LPC
3,0104IBS,C:\Users\judupont\Desktop\AGING_19_01_2026\LNSC\DESHAYES\DESHAYES\DATA_Aging\0104IBS\V4 0104IBS\Ceinture\2019_09_02-10_21_03\2019_09_02-10_21_03_BB_RR.csv,LPC
4,0107LER,C:\Users\judupont\Desktop\AGING_19_01_2026\LNSC\DESHAYES\DESHAYES\DATA_Aging\0107LER\V4 0107LER\Ceinture\No Team Assigned\2019_09_04-10_10_04\2019_09_04-10_10_04_BB_RR.csv,LPC
5,0108MMS,C:\Users\judupont\Desktop\AGING_19_01_2026\LNSC\DESHAYES\DESHAYES\DATA_Aging\0108MMS\V4\Ceinture\2019_10_24-09_06_30_BB_RR.csv,LPC
6,0109MJR,C:\Users\judupont\Desktop\AGING_19_01_2026\LNSC\DESHAYES\DESHAYES\DATA_Aging\0109MJR\V4\Ceinture\2019_09_13-09_33_09_BB_RR.csv,LPC
7,0110RMS,C:\Users\judupont\Desktop\AGING_19_01_2026\LNSC\DESHAYES\DESHAYES\DATA_Aging\0110RMS\V4\Ceinture\2019_09_16-09_30_11_BB_RR.csv,LPC
8,0111TAS,C:\Users\judupont\Desktop\AGING_19_01_2026\LNSC\DESHAYES\DESHAYES\DATA_Aging\0111TAS\V4\Ceinture\2019_10_03-09_26_21_BB_RR.csv,LPC
9,0112DRR,C:\Users\judupont\Desktop\AGING_19_01_2026\LNSC\DESHAYES\DESHAYES\DATA_Aging\0112DRR\V4\Ceinture\2019_11_06-09_35_42_BB_RR.csv,LPC


In [23]:
# ===== Racines à scanner =====
racines = [
    r"C:\Users\judupont\Desktop\data_aging_11_2025_copie\LNSC\LNSC\DESHAYES\DATA_Ancillaire",
    r"C:\Users\judupont\Desktop\AGING_19_01_2026\LPC\GAUTHIER\GAUTHIER\PARTICIPANTS\participants_partie 2_labo"
]

# ===== Numero_inclusion manquants =====
candidats_a_garder = [c.upper() for c in manquants]

# ===== Paramètres poids =====
POIDS_MIN = 2000 * 1024   # 2 Mo
POIDS_MAX = 8000 * 1024   # 8 Mo

# ===== Collecte fichiers =====
rows = []

for racine in racines:
    print(f"\n--- Scan de : {racine} ---")
    
    for root, dirs, files in os.walk(racine):

        if "V4" not in root.upper():
            continue

        for f in files:

            if not f.lower().endswith(".csv"):
                continue
            if not re.search(r"(BB_RR|BR_RR)", f, re.IGNORECASE):
                continue

            chemin = os.path.join(root, f)
            chemin_upper = chemin.upper()

            # ---- filtre poids ----
            try:
                taille = os.path.getsize(chemin)
            except OSError:
                continue
            if not (POIDS_MIN <= taille <= POIDS_MAX):
                continue

            # ---- identification ----
            pid_trouve = None
            for pid in candidats_a_garder:
                # Cas spécial si nécessaire, ex : 0117MSR -> 0117MMR
                if pid == "0117MSR" and "0117MMR" in chemin_upper:
                    pid_trouve = pid
                    break
                if pid in chemin_upper:
                    pid_trouve = pid
                    break

            if not pid_trouve:
                continue

            # ---- ajout à la liste ----
            rows.append({
                "Numero_inclusion": pid_trouve,
                "Chemin": chemin,
                "Feuille": "LPC"
            })
            print(f"OK [{pid_trouve}] :", chemin)

# ===== Création DataFrame =====
df_lpc_manquants = pd.DataFrame(rows)

# ===== Suppression doublons =====
df_lpc_manquants = df_lpc_manquants.drop_duplicates(
    subset="Numero_inclusion",
    keep="first"
).reset_index(drop=True)

# ===== Résumé & contrôle =====
print("\n====================================")
print(f"PIDs recherchés : {len(candidats_a_garder)}")
print(f"Fichiers retrouvés : {len(df_lpc_manquants)}")

manquants_restant = sorted(
    set(candidats_a_garder) - set(df_lpc_manquants["Numero_inclusion"])
)
if manquants_restant:
    print("⚠️ PIDs encore manquants :", manquants_restant)
else:
    print("✅ Tous les PIDs manquants ont été retrouvés")

display(df_lpc_manquants)

# ⚠️ PIDs encore manquants : ['0106DJR', '0137BMS', '0141HJR', '0142LLS'] pas dans l'arbo


--- Scan de : C:\Users\judupont\Desktop\data_aging_11_2025_copie\LNSC\LNSC\DESHAYES\DATA_Ancillaire ---

--- Scan de : C:\Users\judupont\Desktop\AGING_19_01_2026\LPC\GAUTHIER\GAUTHIER\PARTICIPANTS\participants_partie 2_labo ---
OK [0130FAR] : C:\Users\judupont\Desktop\AGING_19_01_2026\LPC\GAUTHIER\GAUTHIER\PARTICIPANTS\participants_partie 2_labo\0130FAR-0154FAR\V4\0130FAR\ceinture\2020_02_11-09_47_09\2020_02_11-09_47_09_BB_RR.csv
OK [0132GAR] : C:\Users\judupont\Desktop\AGING_19_01_2026\LPC\GAUTHIER\GAUTHIER\PARTICIPANTS\participants_partie 2_labo\0132GAR\0132GAR_V4\ceinture\ceinture\2021_02_08-10_03_14_BB_RR.csv
OK [0133BPS] : C:\Users\judupont\Desktop\AGING_19_01_2026\LPC\GAUTHIER\GAUTHIER\PARTICIPANTS\participants_partie 2_labo\0133BPS-0157BPS\0157BPS_V4\ceinture\No Team Assigned\2021_02_09-09_54_43\2021_02_09-09_54_43_BB_RR.csv
OK [0136CJR] : C:\Users\judupont\Desktop\AGING_19_01_2026\LPC\GAUTHIER\GAUTHIER\PARTICIPANTS\participants_partie 2_labo\0136CJR\0136CJR_V4\Ceinture\No Team

,Numero_inclusion,Chemin,Feuille
0,0130FAR,C:\Users\judupont\Desktop\AGING_19_01_2026\LPC\GAUTHIER\GAUTHIER\PARTICIPANTS\participants_partie 2_labo\0130FAR-0154FAR\V4\0130FAR\ceinture\2020_02_11-09_47_09\2020_02_11-09_47_09_BB_RR.csv,LPC
1,0132GAR,C:\Users\judupont\Desktop\AGING_19_01_2026\LPC\GAUTHIER\GAUTHIER\PARTICIPANTS\participants_partie 2_labo\0132GAR\0132GAR_V4\ceinture\ceinture\2021_02_08-10_03_14_BB_RR.csv,LPC
2,0133BPS,C:\Users\judupont\Desktop\AGING_19_01_2026\LPC\GAUTHIER\GAUTHIER\PARTICIPANTS\participants_partie 2_labo\0133BPS-0157BPS\0157BPS_V4\ceinture\No Team Assigned\2021_02_09-09_54_43\2021_02_09-09_54_43_BB_RR.csv,LPC
3,0136CJR,C:\Users\judupont\Desktop\AGING_19_01_2026\LPC\GAUTHIER\GAUTHIER\PARTICIPANTS\participants_partie 2_labo\0136CJR\0136CJR_V4\Ceinture\No Team Assigned\2021_12_10-10_17_58\2021_12_10-10_17_58_BB_RR.csv,LPC
4,0138NWR,C:\Users\judupont\Desktop\AGING_19_01_2026\LPC\GAUTHIER\GAUTHIER\PARTICIPANTS\participants_partie 2_labo\0138NWR\0138NWR-V4\ceinture\No Team Assigned\2022_04_04-09_02_10\2022_04_04-09_02_10_BB_RR.csv,LPC
5,0144ZGR,C:\Users\judupont\Desktop\AGING_19_01_2026\LPC\GAUTHIER\GAUTHIER\PARTICIPANTS\participants_partie 2_labo\0144ZGR\0144ZGR_V4\ceinture\No Team Assigned\2023_04_11-09_08_49\2023_04_11-09_08_49_BB_RR.csv,LPC
6,0139BMR,C:\Users\judupont\Desktop\AGING_19_01_2026\LPC\GAUTHIER\GAUTHIER\PARTICIPANTS\participants_partie 2_labo\0167BMR\0167BMR_V4 0139BMR_V4\ceinture\No Team Assigned\2023_02_17-10_05_43\2023_02_17-10_05_43_BB_RR.csv,LPC
7,0140LMR,C:\Users\judupont\Desktop\AGING_19_01_2026\LPC\GAUTHIER\GAUTHIER\PARTICIPANTS\participants_partie 2_labo\0168LMR 0140LMR\V4 0140LMR\ceinture\No Team Assigned\2010_01_02-23_56_56\2010_01_02-23_56_56_BB_RR.csv,LPC
8,0143EBR,C:\Users\judupont\Desktop\AGING_19_01_2026\LPC\GAUTHIER\GAUTHIER\PARTICIPANTS\participants_partie 2_labo\0171EBR _ 0143EBR\0143EBR_V4\ceinture\No Team Assigned\2023_04_25-09_05_36\2023_04_25-09_05_36_BB_RR.csv,LPC


## SAINTE-MARGUERITE

In [24]:
# ===== Racine SAINTE-MARGUERITE =====
racine_st_marguerite = (
    r"C:\Users\judupont\Desktop\AGING_19_01_2026"
    r"\LPC\GAUTHIER\GAUTHIER\PATIENTS\TOUS_patients_APHM_StMarguerite"
)

# ===== Candidats attendus =====
candidats_sm = [
    c.upper() for c in resultats_globaux["SAINTE-MARGUERITE"]["valides"]
]

rows_sm = []

# =========================================================
# BOUCLE PRINCIPALE
# =========================================================
for root, dirs, files in os.walk(racine_st_marguerite):

    if "V4" not in root.upper():
        continue

    for f in files:

        # CSV BB/BR_RR uniquement
        if not f.lower().endswith(".csv"):
            continue
        if not re.search(r"(BB_RR|BR_RR)", f, re.IGNORECASE):
            continue

        chemin = os.path.join(root, f)
        chemin_upper = chemin.upper()

        # ===== identification =====
        pid = None
        for c in candidats_sm:
            if c in chemin_upper:
                pid = c
                break

        if not pid:
            continue

        rows_sm.append({
            "Numero_inclusion": pid,
            "Chemin": chemin,
            "Feuille": "SAINTE-MARGUERITE"
        })

        print(f"OK [{pid}] :", chemin)

# =========================================================
# DATAFRAME + SUPPRESSION DES DOUBLONS
# =========================================================
df_sm = pd.DataFrame(rows_sm)

df_sm = (
    df_sm
    .drop_duplicates(subset="Numero_inclusion", keep="first")
    .reset_index(drop=True)
)

# =========================================================
# AFFICHAGE
# =========================================================
print("\n====================================")
print(f"Candidats attendus SAINTE-MARGUERITE : {len(candidats_sm)}")
print(f"Fichiers retenus                     : {len(df_sm)}")

manquants = sorted(
    set(candidats_sm) - set(df_sm["Numero_inclusion"])
)

if manquants:
    print("⚠️ Candidats manquants :", manquants)
else:
    print("✅ Tous les candidats SAINTE-MARGUERITE sont présents")

display(df_sm)


OK [0801HDR] : C:\Users\judupont\Desktop\AGING_19_01_2026\LPC\GAUTHIER\GAUTHIER\PATIENTS\TOUS_patients_APHM_StMarguerite\0801HDR\0801HDR - V4\Ceinture\2022_12_07-11_15_45_BB_RR.csv
OK [0802LAS] : C:\Users\judupont\Desktop\AGING_19_01_2026\LPC\GAUTHIER\GAUTHIER\PATIENTS\TOUS_patients_APHM_StMarguerite\0802LAS\0802LAS - V4\Ceinture\2022_12_12-11_03_54_BB_RR.csv
OK [0803DPS] : C:\Users\judupont\Desktop\AGING_19_01_2026\LPC\GAUTHIER\GAUTHIER\PATIENTS\TOUS_patients_APHM_StMarguerite\0803DPS\0803DPS-V4\ceinture\No Team Assigned\2023_03_01-10_20_00\2023_03_01-10_20_00_BB_RR.csv
OK [0804GOR] : C:\Users\judupont\Desktop\AGING_19_01_2026\LPC\GAUTHIER\GAUTHIER\PATIENTS\TOUS_patients_APHM_StMarguerite\0804GOR\0804GOR_V4\Ceinture\No Team Assigned\2023_01_11-09_53_11\2023_01_11-09_53_11_BB_RR.csv
OK [0805BMS] : C:\Users\judupont\Desktop\AGING_19_01_2026\LPC\GAUTHIER\GAUTHIER\PATIENTS\TOUS_patients_APHM_StMarguerite\0805BMS\0805BMS - V4\Ceinture\No Team Assigned\2023_01_18-09_11_15\2023_01_18-09_11_1

,Numero_inclusion,Chemin,Feuille
0,0801HDR,C:\Users\judupont\Desktop\AGING_19_01_2026\LPC\GAUTHIER\GAUTHIER\PATIENTS\TOUS_patients_APHM_StMarguerite\0801HDR\0801HDR - V4\Ceinture\2022_12_07-11_15_45_BB_RR.csv,SAINTE-MARGUERITE
1,0802LAS,C:\Users\judupont\Desktop\AGING_19_01_2026\LPC\GAUTHIER\GAUTHIER\PATIENTS\TOUS_patients_APHM_StMarguerite\0802LAS\0802LAS - V4\Ceinture\2022_12_12-11_03_54_BB_RR.csv,SAINTE-MARGUERITE
2,0803DPS,C:\Users\judupont\Desktop\AGING_19_01_2026\LPC\GAUTHIER\GAUTHIER\PATIENTS\TOUS_patients_APHM_StMarguerite\0803DPS\0803DPS-V4\ceinture\No Team Assigned\2023_03_01-10_20_00\2023_03_01-10_20_00_BB_RR.csv,SAINTE-MARGUERITE
3,0804GOR,C:\Users\judupont\Desktop\AGING_19_01_2026\LPC\GAUTHIER\GAUTHIER\PATIENTS\TOUS_patients_APHM_StMarguerite\0804GOR\0804GOR_V4\Ceinture\No Team Assigned\2023_01_11-09_53_11\2023_01_11-09_53_11_BB_RR.csv,SAINTE-MARGUERITE
4,0805BMS,C:\Users\judupont\Desktop\AGING_19_01_2026\LPC\GAUTHIER\GAUTHIER\PATIENTS\TOUS_patients_APHM_StMarguerite\0805BMS\0805BMS - V4\Ceinture\No Team Assigned\2023_01_18-09_11_15\2023_01_18-09_11_15_BB_RR.csv,SAINTE-MARGUERITE
5,0806KHS,C:\Users\judupont\Desktop\AGING_19_01_2026\LPC\GAUTHIER\GAUTHIER\PATIENTS\TOUS_patients_APHM_StMarguerite\0806KHS\0806KHS-V4\Ceinture\No Team Assigned\2023_03_27-08_28_46\2023_03_27-08_28_46_BB_RR.csv,SAINTE-MARGUERITE
6,0807OMR,C:\Users\judupont\Desktop\AGING_19_01_2026\LPC\GAUTHIER\GAUTHIER\PATIENTS\TOUS_patients_APHM_StMarguerite\0807OMR\V4\ceinture\No Team Assigned\2023_05_15-09_39_03\2023_05_15-09_39_03_BB_RR.csv,SAINTE-MARGUERITE


## CGD

In [26]:
# ===== Racine CGD =====
racine_cgd = (
    r"C:\Users\judupont\Desktop\AGING_19_01_2026"
    r"\LPC\GAUTHIER\GAUTHIER\PATIENTS\TOUS_patients_CGD"
)

# ===== Candidats attendus =====
candidats_cgd = [
    c.upper() for c in resultats_globaux["CGD"]["valides"]
]

rows_cgd = []

# =========================================================
# BOUCLE PRINCIPALE
# =========================================================
for root, dirs, files in os.walk(racine_cgd):

    if "V4" not in root.upper():
        continue

    for f in files:

        if not f.lower().endswith(".csv"):
            continue
        if not re.search(r"(BB_RR|BR_RR)", f, re.IGNORECASE):
            continue

        chemin = os.path.join(root, f)
        chemin_upper = chemin.upper()

        # ===== identification =====
        pid = None
        for c in candidats_cgd:
            if c in chemin_upper:
                pid = c
                break

        if not pid:
            continue

        rows_cgd.append({
            "Numero_inclusion": pid,
            "Chemin": chemin,
            "Feuille": "CGD"
        })

        print(f"OK [{pid}] :", chemin)

# =========================================================
# DATAFRAME + SUPPRESSION DES DOUBLONS
# =========================================================
df_cgd = pd.DataFrame(rows_cgd)

df_cgd = (
    df_cgd
    .drop_duplicates(subset="Numero_inclusion", keep="first")
    .reset_index(drop=True)
)

# =========================================================
# AFFICHAGE
# =========================================================
print("\n====================================")
print(f"Candidats attendus CGD : {len(candidats_cgd)}")
print(f"Fichiers retenus CGD   : {len(df_cgd)}")

manquants = sorted(
    set(candidats_cgd) - set(df_cgd["Numero_inclusion"])
)

if manquants:
    print("⚠️ Candidats manquants :", manquants)
else:
    print("✅ Tous les candidats CGD sont présents")

display(df_cgd)


OK [0901SMR] : C:\Users\judupont\Desktop\AGING_19_01_2026\LPC\GAUTHIER\GAUTHIER\PATIENTS\TOUS_patients_CGD\0901SMR\0901SMR - V4\Ceinture\2022_12_14-10_53_43_BB_RR.csv
OK [0903DJS] : C:\Users\judupont\Desktop\AGING_19_01_2026\LPC\GAUTHIER\GAUTHIER\PATIENTS\TOUS_patients_CGD\0903DJS\V4\ceinture\No Team Assigned\2023_03_08-10_03_10\2023_03_08-10_03_10_BB_RR.csv
OK [0904KJS] : C:\Users\judupont\Desktop\AGING_19_01_2026\LPC\GAUTHIER\GAUTHIER\PATIENTS\TOUS_patients_CGD\0904KJS\0904KJS-V4\ceinture\No Team Assigned\2023_03_22-09_03_56\2023_03_22-09_03_56_BB_RR.csv
OK [0905SLR] : C:\Users\judupont\Desktop\AGING_19_01_2026\LPC\GAUTHIER\GAUTHIER\PATIENTS\TOUS_patients_CGD\0905SLR\0905SLR-V4\ceinture\No Team Assigned\2023_04_05-09_51_00\2023_04_05-09_51_00_BB_RR.csv

Candidats attendus CGD : 5
Fichiers retenus CGD   : 4
⚠️ Candidats manquants : ['0902SIS']


,Numero_inclusion,Chemin,Feuille
0,0901SMR,C:\Users\judupont\Desktop\AGING_19_01_2026\LPC\GAUTHIER\GAUTHIER\PATIENTS\TOUS_patients_CGD\0901SMR\0901SMR - V4\Ceinture\2022_12_14-10_53_43_BB_RR.csv,CGD
1,0903DJS,C:\Users\judupont\Desktop\AGING_19_01_2026\LPC\GAUTHIER\GAUTHIER\PATIENTS\TOUS_patients_CGD\0903DJS\V4\ceinture\No Team Assigned\2023_03_08-10_03_10\2023_03_08-10_03_10_BB_RR.csv,CGD
2,0904KJS,C:\Users\judupont\Desktop\AGING_19_01_2026\LPC\GAUTHIER\GAUTHIER\PATIENTS\TOUS_patients_CGD\0904KJS\0904KJS-V4\ceinture\No Team Assigned\2023_03_22-09_03_56\2023_03_22-09_03_56_BB_RR.csv,CGD
3,0905SLR,C:\Users\judupont\Desktop\AGING_19_01_2026\LPC\GAUTHIER\GAUTHIER\PATIENTS\TOUS_patients_CGD\0905SLR\0905SLR-V4\ceinture\No Team Assigned\2023_04_05-09_51_00\2023_04_05-09_51_00_BB_RR.csv,CGD


## CERCA

In [28]:
import os
import re
import pandas as pd

# ===== Racines CERCA =====
racines_cerca = [
    r"C:\Users\judupont\Desktop\AGING_19_01_2026\CERCA\GUILLOU\GUILLOU\CeRCA POITIERS",
    r"C:\Users\judupont\Desktop\AGING_19_01_2026\CERCA\RIGALLEAU\RIGALLEAU"
]

# ===== Candidats CERCA =====
candidats_cerca = [c.upper() for c in resultats_globaux["CERCA"]["valides"]]

fichiers_cerca = []

for racine in racines_cerca:
    print(f"\n--- Scan CERCA : {racine} ---")

    for root, dirs, files in os.walk(racine):

        if "V4" not in root.upper():
            continue

        for f in files:
            if not f.lower().endswith(".csv"):
                continue
            if not re.search(r"(BB_RR|BR_RR)", f, re.IGNORECASE):
                continue

            chemin = os.path.join(root, f)
            chemin_upper = chemin.upper()

            for pid in candidats_cerca:
                if pid in chemin_upper:
                    fichiers_cerca.append((pid, chemin))
                    print(f"OK [{pid}] :", chemin)
                    break

rows = []

for pid in candidats_cerca:
    fichiers_pid = [c for p, c in fichiers_cerca if p == pid]

    if not fichiers_pid:
        continue

    ceinture = [f for f in fichiers_pid if "CEINTURE" in f.upper()]
    chemin_final = ceinture[0] if ceinture else fichiers_pid[0]

    rows.append({
        "Numero_inclusion": pid,
        "Chemin": chemin_final,
        "Feuille": "CERCA"
    })

df_cerca = pd.DataFrame(rows)

# ===== Affichage =====
print("\n====================================")
print(f"Candidats attendus CERCA : {len(candidats_cerca)}")
print(f"Fichiers BB/BR_RR V4 retenus : {len(df_cerca)}")

manquants = sorted(set(candidats_cerca) - set(df_cerca["Numero_inclusion"]))
if manquants:
    print("⚠️ Candidats CERCA manquants :", manquants)
else:
    print("✅ Tous les candidats CERCA sont présents")

display(df_cerca)



--- Scan CERCA : C:\Users\judupont\Desktop\AGING_19_01_2026\CERCA\GUILLOU\GUILLOU\CeRCA POITIERS ---
OK [0401BDS] : C:\Users\judupont\Desktop\AGING_19_01_2026\CERCA\GUILLOU\GUILLOU\CeRCA POITIERS\Alizé GRANGE\0401BDS\V4\Ceinture\2019_11_13__09_37_08_BR_RR.csv
OK [0405RCR] : C:\Users\judupont\Desktop\AGING_19_01_2026\CERCA\GUILLOU\GUILLOU\CeRCA POITIERS\Alizé GRANGE\0405RCR\V4\Ceinture\2020_02_14__10_06_12_BR_RR.csv
OK [0406BBS] : C:\Users\judupont\Desktop\AGING_19_01_2026\CERCA\GUILLOU\GUILLOU\CeRCA POITIERS\Alizé GRANGE\0406BBS\V4\Ceinture\2019_12_18__11_01_15_BR_RR.csv
OK [0408SJS] : C:\Users\judupont\Desktop\AGING_19_01_2026\CERCA\GUILLOU\GUILLOU\CeRCA POITIERS\Alizé GRANGE\0408SJS\V4\Ceinture\2019_11_27__10_38_19_BR_RR.csv
OK [0404DER] : C:\Users\judupont\Desktop\AGING_19_01_2026\CERCA\GUILLOU\GUILLOU\CeRCA POITIERS\Camille GUILLOU\0404DER\V4\Ceinture\2019_11_08__09_47_26_BR_RR.csv
OK [0409HCR] : C:\Users\judupont\Desktop\AGING_19_01_2026\CERCA\GUILLOU\GUILLOU\CeRCA POITIERS\Camil

,Numero_inclusion,Chemin,Feuille
0,0401BDS,C:\Users\judupont\Desktop\AGING_19_01_2026\CERCA\GUILLOU\GUILLOU\CeRCA POITIERS\Alizé GRANGE\0401BDS\V4\Ceinture\2019_11_13__09_37_08_BR_RR.csv,CERCA
1,0404DER,C:\Users\judupont\Desktop\AGING_19_01_2026\CERCA\GUILLOU\GUILLOU\CeRCA POITIERS\Camille GUILLOU\0404DER\V4\Ceinture\2019_11_08__09_47_26_BR_RR.csv,CERCA
2,0405RCR,C:\Users\judupont\Desktop\AGING_19_01_2026\CERCA\GUILLOU\GUILLOU\CeRCA POITIERS\Alizé GRANGE\0405RCR\V4\Ceinture\2020_02_14__10_06_12_BR_RR.csv,CERCA
3,0406BBS,C:\Users\judupont\Desktop\AGING_19_01_2026\CERCA\GUILLOU\GUILLOU\CeRCA POITIERS\Alizé GRANGE\0406BBS\V4\Ceinture\2019_12_18__11_01_15_BR_RR.csv,CERCA
4,0408SJS,C:\Users\judupont\Desktop\AGING_19_01_2026\CERCA\GUILLOU\GUILLOU\CeRCA POITIERS\Alizé GRANGE\0408SJS\V4\Ceinture\2019_11_27__10_38_19_BR_RR.csv,CERCA
5,0409HCR,C:\Users\judupont\Desktop\AGING_19_01_2026\CERCA\GUILLOU\GUILLOU\CeRCA POITIERS\Camille GUILLOU\0409HCR\V4\Ceinture\2020_01_06__10_24_59_BR_RR.csv,CERCA
6,0410FMS,C:\Users\judupont\Desktop\AGING_19_01_2026\CERCA\GUILLOU\GUILLOU\CeRCA POITIERS\Camille GUILLOU\0410FMS\V4\Ceinture\2019_11_18__10_30_15_BR_RR.csv,CERCA
7,0414PVR,C:\Users\judupont\Desktop\AGING_19_01_2026\CERCA\RIGALLEAU\RIGALLEAU\participant Sarah\0414PVR\V4\Ceinture\2021_07_09__09_38_01_BR_RR.csv,CERCA
8,0420MCR,C:\Users\judupont\Desktop\AGING_19_01_2026\CERCA\RIGALLEAU\RIGALLEAU\participant Sarah\0420MCR\V4\Ceinture\2021_07_05__14_07_01_BR_RR.csv,CERCA


## CONCATENATION

In [29]:
# ===== Liste des DataFrames par feuille =====
dfs = [
    df_aphm_final,  
    df_caen_final,
    df_poitiers_final,     
    df_rouen_final,
    df_laveran_final,
    df_lpc_final,         
    df_lpc_manquants,
    df_sm,          
    df_cgd,         
    df_cerca      
]

# ===== Normalisation des colonnes =====
for i, df in enumerate(dfs):
    if "PID" in df.columns:
        df.rename(columns={"PID": "Numero_inclusion"}, inplace=True)
    # Assurer que la colonne Condition existe
    if "Condition" not in df.columns:
        df["Condition"] = None
    # On garde uniquement les colonnes essentielles
    df = df[["Numero_inclusion", "Chemin", "Feuille"]]
    dfs[i] = df

# ===== Concatenation =====
df_global = pd.concat(dfs, ignore_index=True)

# ===== Suppression des doublons par Numero_inclusion =====
# On garde le premier fichier détecté par candidat
df_global = df_global.drop_duplicates(subset=["Numero_inclusion"], keep="first")

# ===== Tri par Numero_inclusion =====
df_global = df_global.sort_values("Numero_inclusion").reset_index(drop=True)

# ===== Résumé =====
print(f"Total candidats uniques : {df_global['Numero_inclusion'].nunique()}")
print(f"Total fichiers conservés : {len(df_global)}")
display(df_global)


Total candidats uniques : 134
Total fichiers conservés : 134


,Numero_inclusion,Chemin,Feuille
0,0101CAR,C:\Users\judupont\Desktop\AGING_19_01_2026\LPC\GAUTHIER\GAUTHIER\PATIENTS\TOUS_patients_APHM_laveran\0101CAR\0101CAR 04-04-2019 V4\Ceinture 0101CAR\2019_04_04-08_34_38_BB_RR.csv,APHM
1,0101EMS,C:\Users\judupont\Desktop\AGING_19_01_2026\LNSC\DESHAYES\DESHAYES\DATA_Aging\0101EMS\0101EMS V4\Ceinture\2019_10_22-09_42_52\2019_10_22-09_42_52_BB_RR.csv,LPC
2,0102EMS,C:\Users\judupont\Desktop\AGING_19_01_2026\LNSC\DESHAYES\DESHAYES\DATA_Aging\0102EMS\V4 0102EMS\Ceinture\2019_09_05-09_36_48\2019_09_05-09_36_48_BB_RR.csv,LPC
3,0102PCR,C:\Users\judupont\Desktop\AGING_19_01_2026\LPC\GAUTHIER\GAUTHIER\PATIENTS\TOUS_patients_APHM_laveran\0102PCR\0102PCR 12-04-2019 V4\Ceinture 0102PCR\2019_04_12-09_06_05_BB_RR.csv,APHM
4,0103BPS,C:\Users\judupont\Desktop\AGING_19_01_2026\LNSC\DESHAYES\DESHAYES\DATA_Aging\0103BPS\V4 0103BPS\Ceinture\2019_09_17-09_43_26\2019_09_17-09_43_26_BB_RR.csv,LPC
5,0104FJS,C:\Users\judupont\Desktop\AGING_19_01_2026\LPC\GAUTHIER\GAUTHIER\PATIENTS\TOUS_patients_APHM_laveran\0104FJS\0104FJS 08-11-2019 V4\Ceinture 0104FJS\2019_11_08-08_26_41_BB_RR.csv,APHM
6,0104IBS,C:\Users\judupont\Desktop\AGING_19_01_2026\LNSC\DESHAYES\DESHAYES\DATA_Aging\0104IBS\V4 0104IBS\Ceinture\2019_09_02-10_21_03\2019_09_02-10_21_03_BB_RR.csv,LPC
7,0105PNR,C:\Users\judupont\Desktop\AGING_19_01_2026\LPC\GAUTHIER\GAUTHIER\PATIENTS\TOUS_patients_APHM_laveran\0105PNR\0105PNR 09-09-2019 V4\Ceinture 0105PNR\2019_09_09-08_53_04_BB_RR.csv,APHM
8,0107DSS,C:\Users\judupont\Desktop\AGING_19_01_2026\LPC\GAUTHIER\GAUTHIER\PATIENTS\TOUS_patients_APHM_laveran\0107DSS\0107DSS 17-10-2019 V4\Ceinture 0107DSS\2019_10_17-09_08_45_BB_RR.csv,APHM
9,0107LER,C:\Users\judupont\Desktop\AGING_19_01_2026\LNSC\DESHAYES\DESHAYES\DATA_Aging\0107LER\V4 0107LER\Ceinture\No Team Assigned\2019_09_04-10_10_04\2019_09_04-10_10_04_BB_RR.csv,LPC


In [30]:
# CONSERVATION DES FEUILLES ET CONDITIONS DANS UN FICHIER TXT 

chemin_txt = r"C:\Users\judupont\Desktop\df_global_v4.txt"

df_global.to_csv(
    chemin_txt,
    sep="\t",
    index=False,
    encoding="utf-8-sig"
)

print(f"📄 df_global sauvegardé : {chemin_txt}")

📄 df_global sauvegardé : C:\Users\judupont\Desktop\df_global_v4.txt


# Verification que le temps d'experience et le temps du fichier BB_RR correspondent, offset et continuité

In [31]:
def convertir_heure_en_secondes(x):
    if pd.isna(x):
        return None

    # Excel → heure uniquement
    if isinstance(x, (int, float)):
        seconds = float(x) * 24 * 3600
        return seconds % (24 * 3600)

    try:
        t = pd.to_datetime(str(x).strip()).time()
        return t.hour * 3600 + t.minute * 60 + t.second
    except Exception:
        return None



dfs_excel = []

for feuille, data in resultats_bloc1.items():

    df_excel = data["df"].copy()

    # filtrage candidats retenus
    df_excel["Numero_inclusion"] = df_excel["Numero_inclusion"].astype(str)
    df_excel = df_excel[
        df_excel["Numero_inclusion"].isin(df_global['Numero_inclusion'])
    ]

    if df_excel.empty:
        continue

    # Conversion heures → secondes
    df_excel["heure_debut_sec"] = df_excel["heure_ceinture_v4"].apply(convertir_heure_en_secondes)
    df_excel["heure_fin_sec"] = df_excel["heure_fin_tests_v4"].apply(convertir_heure_en_secondes)

    # Durée visite Excel
    df_excel["duree_experience_sec"] = (
        df_excel["heure_fin_sec"] - df_excel["heure_debut_sec"]
    )

    # Sécurité passage minuit
    df_excel.loc[
        df_excel["duree_experience_sec"] < 0,
        "duree_experience_sec"
    ] += 24 * 3600

    df_excel["Feuille"] = feuille
    dfs_excel.append(df_excel)

df_excel_global = pd.concat(dfs_excel, ignore_index=True)


print(f"Lignes Excel retenues : {len(df_excel_global)}")
display(df_excel_global[[
    "Numero_inclusion",
    "Feuille",
    "duree_experience_sec"
]])

Lignes Excel retenues : 134


,Numero_inclusion,Feuille,duree_experience_sec
0,0101CAR,APHM,9660
1,0102PCR,APHM,7440
2,0105PNR,APHM,7140
3,0104FJS,APHM,6180
4,0107DSS,APHM,6060
5,0109GSS,APHM,7140
6,0110LPR,APHM,8760
7,0111MNR,APHM,7620
8,0112BSR,APHM,8220
9,0114LMS,APHM,6960


In [32]:
# ==== Conversion Date dans Excel ====
df_excel_global["Date_v4_dt"] = pd.to_datetime(
    df_excel_global["Date_v4"], errors="coerce"
).dt.date

# ==== Liste des candidats pour lesquels on ignore la vérification de la date car verifier à la main ==== # 0140LMR trop abberant
ignore_date_check = ["0114MLR","0503LCR","0512RCR","0513EBS"]

rows_rr = []

for _, row in df_global.iterrows():

    pid = str(row["Numero_inclusion"])
    chemin = row["Chemin"]

    # ---- récupérer la ligne Excel correspondant au PID ----
    ligne_excel = df_excel_global[df_excel_global["Numero_inclusion"] == pid]
    if ligne_excel.empty:
        continue

    date_v4 = ligne_excel.iloc[0]["Date_v4_dt"]
    duree_excel_sec = ligne_excel.iloc[0]["duree_experience_sec"]

    # ---- lecture CSV BB_RR ----
    try:
        df_rr = pd.read_csv(chemin)
    except Exception:
        continue

    if "Timestamp" not in df_rr.columns:
        continue

    # ---- conversion timestamp CSV (ignorer les millisecondes) ----
    df_rr["Timestamp_dt"] = pd.to_datetime(
        df_rr["Timestamp"].str.split(".").str[0],
        dayfirst=True,
        errors="coerce"
    )
    df_rr = df_rr.dropna(subset=["Timestamp_dt"])

    # ---- filtrage par date si pas dans ignore_date_check ----
    if pid not in ignore_date_check and pd.notna(date_v4):
        df_rr = df_rr[df_rr["Timestamp_dt"].dt.date == date_v4]

    if df_rr.empty:
        continue

    # ---- durée RR (sans ms) ----
    t_min = df_rr["Timestamp_dt"].dt.floor("s").min()
    t_max = df_rr["Timestamp_dt"].dt.floor("s").max()
    duree_rr_sec = (t_max - t_min).total_seconds()

    rows_rr.append({
        "Numero_inclusion": pid,
        "duree_rr_sec": duree_rr_sec,
        "duree_excel_sec": duree_excel_sec,
        "nb_points": len(df_rr),
    })

df_rr_global = pd.DataFrame(rows_rr)
print(f"RR calculés : {len(df_rr_global)}")
display(df_rr_global)


RR calculés : 133


,Numero_inclusion,duree_rr_sec,duree_excel_sec,nb_points
0,0101CAR,9809.0,9660,175160
1,0101EMS,8011.0,7860,143053
2,0102EMS,4897.0,4980,87446
3,0102PCR,7699.0,7440,137482
4,0103BPS,7069.0,7020,126232
5,0104FJS,6386.0,6180,114035
6,0104IBS,6995.0,6900,124910
7,0105PNR,7210.0,7140,128750
8,0107DSS,6228.0,6060,111214
9,0107LER,6890.0,6780,123035


# D'ou vient l'écart ? L'offest doit etre appliqué en debut, en fin, au milieu?

In [33]:
def sec_to_datetime(sec, date_ref):
    return datetime.combine(
        date_ref.date(),
        datetime.min.time()
    ) + timedelta(seconds=int(sec))


resultats_offset = []

# apres verification à la main ces candidats ont leur heure de debut decaler de plus ou moins 1h donc on decale pour appliquer le découpage quand meme

candidats_rebase_excel = {
    "0116VAR", "0138NWR", "0140LMR", "0143EBR", "0144ZGR", "0626MCS",
    "0310ACS", "0329NMR", "0806KHS", "0339NPR","0801HDR", "0802LAS", "0901SMR"
}


for _, row in df_global.iterrows():

    pid = str(row["Numero_inclusion"]).strip().upper()
    chemin_bb_rr = row["Chemin"]

    # ===== Lecture BB_RR (ZIP ou non) =====
    try:
        if ".ZIP|" in chemin_bb_rr.upper():
            zip_path, inner_csv = chemin_bb_rr.split("|", 1)
            with zipfile.ZipFile(zip_path, "r") as z:
                with z.open(inner_csv) as f:
                    df_rr = pd.read_csv(f)
        else:
            df_rr = pd.read_csv(chemin_bb_rr)
    except Exception as e:
        print(f"⚠️ Lecture BB_RR impossible pour {pid} : {e}")
        continue

    if "Timestamp" not in df_rr.columns:
        continue

    # ===== Parsing timestamp =====
    df_rr["Timestamp"] = pd.to_datetime(
        df_rr["Timestamp"].astype(str).str.split(".").str[0],
        dayfirst=True,
        errors="coerce"
    )
    df_rr = df_rr.dropna(subset=["Timestamp"])
    if df_rr.empty:
        continue

    t_min = df_rr["Timestamp"].min()
    t_max = df_rr["Timestamp"].max()

    # ===== Excel =====
    ligne_excel = df_excel_global.loc[
        df_excel_global["Numero_inclusion"] == pid
    ]
    if ligne_excel.empty:
        continue

    duree_excel_sec = ligne_excel["duree_experience_sec"].values[0]
    if pd.isna(duree_excel_sec):
        continue

    # ===== Cas spécial : rebase Excel sur RR =====
    if pid in candidats_rebase_excel:
        h_debut = t_min
        h_fin = h_debut + pd.to_timedelta(duree_excel_sec, unit="s")
        mode = "REBASE_RR"
        # 🔹 delta_retard_sec pour info, même si rebase
        delta_retard_sec = 0
    else:
        h_debut_sec = ligne_excel["heure_debut_sec"].values[0]
        h_fin_sec = ligne_excel["heure_fin_sec"].values[0]
        if pd.isna(h_debut_sec) or pd.isna(h_fin_sec):
            continue

        h_debut = sec_to_datetime(h_debut_sec, t_min)
        h_fin   = sec_to_datetime(h_fin_sec, t_min)
        mode = "STANDARD"

        # Calcul du delta retard sec
        delta_retard_sec = max((t_min - h_debut).total_seconds(), 0)

    # ===== Comptages =====
    nb_avant = (df_rr["Timestamp"] < h_debut).sum()
    nb_dans = ((df_rr["Timestamp"] >= h_debut) & (df_rr["Timestamp"] <= h_fin)).sum()
    nb_apres = (df_rr["Timestamp"] > h_fin).sum()
    total = len(df_rr)

    # ===== Diagnostic =====
    if nb_apres > nb_avant:
        origine = "FIN"
    elif nb_avant > nb_apres:
        origine = "DEBUT"
    else:
        origine = "MIXTE"

    resultats_offset.append({
        "Numero_inclusion": pid,
        "mode_calcul": mode,
        "nb_total_points": total,
        "nb_avant_excel": nb_avant,
        "nb_dans_excel": nb_dans,
        "nb_apres_excel": nb_apres,
        "origine_offset": origine,
        "delta_retard_sec": delta_retard_sec  
    })

df_offset_bb_rr = pd.DataFrame(resultats_offset)

print(f"Offsets BB_RR calculés : {len(df_offset_bb_rr)}")
display(df_offset_bb_rr)

Offsets BB_RR calculés : 134


,Numero_inclusion,mode_calcul,nb_total_points,nb_avant_excel,nb_dans_excel,nb_apres_excel,origine_offset,delta_retard_sec
0,0101CAR,STANDARD,175160,0,171833,3327,FIN,38.0
1,0101EMS,STANDARD,143053,136,140375,2542,FIN,0.0
2,0102EMS,STANDARD,87446,0,87446,0,MIXTE,48.0
3,0102PCR,STANDARD,137482,0,132779,4703,FIN,5.0
4,0103BPS,STANDARD,126232,0,124904,1328,FIN,26.0
5,0104FJS,STANDARD,114035,332,110375,3328,FIN,0.0
6,0104IBS,STANDARD,124910,0,123171,1739,FIN,3.0
7,0105PNR,STANDARD,128750,0,127439,1311,FIN,4.0
8,0107DSS,STANDARD,111214,261,108232,2721,FIN,0.0
9,0107LER,STANDARD,123035,0,121010,2025,FIN,4.0


In [34]:
from pathlib import Path
import pandas as pd
import os

# ===== Dossier de sortie =====
desktop = Path.home() / "Desktop"
output_dir = desktop / "bb_rr_tronqués_v4"
output_dir.mkdir(exist_ok=True)

print(f"📁 Dossier de sortie : {output_dir}")

# ===== Boucle sur les fichiers tronqués =====
for _, row in df_offset_bb_rr.iterrows():

    pid = row["Numero_inclusion"]

    # récupérer chemin BB_RR depuis df_global
    ligne_global = df_global[df_global["Numero_inclusion"] == pid]
    if ligne_global.empty:
        continue

    chemin_bb_rr = ligne_global.iloc[0]["Chemin"]

    nb_avant = int(row["nb_avant_excel"])
    nb_apres = int(row["nb_apres_excel"])

    # ===== Lecture BB_RR =====
    try:
        df_rr = pd.read_csv(chemin_bb_rr)
    except Exception as e:
        print(f"❌ Lecture impossible {pid}: {e}")
        continue

    total = len(df_rr)

    # ===== Indices de coupe =====
    start = nb_avant
    end = total - nb_apres

    if start >= end:
        print(f"⚠️ Troncature invalide pour {pid} (start={start}, end={end})")
        continue

    df_rr_trunc = df_rr.iloc[start:end].reset_index(drop=True)

    if df_rr_trunc.empty:
        print(f"⚠️ {pid} → fichier vide après troncature")
        continue

    # ===== Nom fichier =====
    nom_fichier = f"offset_{pid}_{Path(chemin_bb_rr).name}"
    path_sortie = output_dir / nom_fichier

    # ===== Sauvegarde =====
    df_rr_trunc.to_csv(path_sortie, index=False)

    print(
        f"✅ {pid} | "
        f"avant={nb_avant}, après={nb_apres} | "
        f"{len(df_rr_trunc)} lignes → {path_sortie.name}"
    )

print("🎯 Troncature terminée")


📁 Dossier de sortie : C:\Users\judupont\Desktop\bb_rr_tronqués_v4
✅ 0101CAR | avant=0, après=3327 | 171833 lignes → offset_0101CAR_2019_04_04-08_34_38_BB_RR.csv
✅ 0101EMS | avant=136, après=2542 | 140375 lignes → offset_0101EMS_2019_10_22-09_42_52_BB_RR.csv
✅ 0102EMS | avant=0, après=0 | 87446 lignes → offset_0102EMS_2019_09_05-09_36_48_BB_RR.csv
✅ 0102PCR | avant=0, après=4703 | 132779 lignes → offset_0102PCR_2019_04_12-09_06_05_BB_RR.csv
✅ 0103BPS | avant=0, après=1328 | 124904 lignes → offset_0103BPS_2019_09_17-09_43_26_BB_RR.csv
✅ 0104FJS | avant=332, après=3328 | 110375 lignes → offset_0104FJS_2019_11_08-08_26_41_BB_RR.csv
✅ 0104IBS | avant=0, après=1739 | 123171 lignes → offset_0104IBS_2019_09_02-10_21_03_BB_RR.csv
✅ 0105PNR | avant=0, après=1311 | 127439 lignes → offset_0105PNR_2019_09_09-08_53_04_BB_RR.csv
✅ 0107DSS | avant=261, après=2721 | 108232 lignes → offset_0107DSS_2019_10_17-09_08_45_BB_RR.csv
✅ 0107LER | avant=0, après=2025 | 121010 lignes → offset_0107LER_2019_09_04-1

In [35]:
# ===== Paramètres =====
min_lignes = 70000

dossier = Path.home() / "Desktop" / "bb_rr_tronqués_v4"

print(f"📂 Dossier analysé : {dossier}")

supprimes = []
conserves = []

for fichier in dossier.glob("*.csv"):

    nom = fichier.name

    # ===== Lecture rapide =====
    try:
        n_lignes = sum(1 for _ in open(fichier, "r", encoding="utf-8")) - 1
    except Exception as e:
        print(f"❌ Lecture impossible {nom}: {e}")
        continue

    # ===== Exclusion fichiers trop courts =====
    if n_lignes < min_lignes:
        fichier.unlink()
        supprimes.append((nom, f"{n_lignes} lignes"))
        print(f"🗑️ {nom} → trop petit ({n_lignes} lignes)")
    else:
        conserves.append((nom, n_lignes))
        print(f"✅ {nom} → conservé ({n_lignes} lignes)")

print("\n====== RÉSUMÉ ======")
print(f"Fichiers supprimés : {len(supprimes)}")
print(f"Fichiers conservés : {len(conserves)}")


📂 Dossier analysé : C:\Users\judupont\Desktop\bb_rr_tronqués_v4
✅ offset_0101CAR_2019_04_04-08_34_38_BB_RR.csv → conservé (171833 lignes)
✅ offset_0101EMS_2019_10_22-09_42_52_BB_RR.csv → conservé (140375 lignes)
✅ offset_0102EMS_2019_09_05-09_36_48_BB_RR.csv → conservé (87446 lignes)
✅ offset_0102PCR_2019_04_12-09_06_05_BB_RR.csv → conservé (132779 lignes)
✅ offset_0103BPS_2019_09_17-09_43_26_BB_RR.csv → conservé (124904 lignes)
✅ offset_0104FJS_2019_11_08-08_26_41_BB_RR.csv → conservé (110375 lignes)
✅ offset_0104IBS_2019_09_02-10_21_03_BB_RR.csv → conservé (123171 lignes)
✅ offset_0105PNR_2019_09_09-08_53_04_BB_RR.csv → conservé (127439 lignes)
✅ offset_0107DSS_2019_10_17-09_08_45_BB_RR.csv → conservé (108232 lignes)
✅ offset_0107LER_2019_09_04-10_10_04_BB_RR.csv → conservé (121010 lignes)
✅ offset_0108MMS_2019_10_24-09_06_30_BB_RR.csv → conservé (104475 lignes)
✅ offset_0109GSS_2019_11_14-09_36_14_BB_RR.csv → conservé (120964 lignes)
✅ offset_0109MJR_2019_09_13-09_33_09_BB_RR.csv → 

# On peut passer au découpage!

In [36]:
# Calcul de la durée de l'écriture pour shift le debut du bloc 2, 7 minutes pour les candidats ayant un NR dans l'une des deux colonnes de calcul 

# ===== PARAMÈTRE =====
DUREE_MOYENNE_ECRITURE_SEC = 7 * 60  # 7 minutes

col_debut = "heure_anamnese_fin_v4"
col_fin   = "heure_rlri16imm_debut_v4"

df = df_excel_global.copy()

# ===== Détection valeur non horaire =====
def est_non_horaire(val):
    if pd.isna(val):
        return True
    if isinstance(val, str):
        v = val.strip().upper()
        return v in {"NR", "N/R", "NON RENSEIGNE", ""}
    return False

# ===== Conversion robuste vers Timestamp =====
def convertir_heure(val):
    if est_non_horaire(val):
        return pd.NaT

    # Excel numérique (fraction de jour)
    if isinstance(val, (int, float)):
        seconds = (float(val) * 24 * 3600) % (24 * 3600)
        return pd.Timestamp("1900-01-01") + pd.to_timedelta(seconds, unit="s")

    # datetime / Timestamp
    if isinstance(val, (pd.Timestamp, datetime.datetime)):
        return pd.Timestamp(
            year=1900, month=1, day=1,
            hour=val.hour, minute=val.minute, second=val.second
        )

    # texte
    try:
        t = pd.to_datetime(str(val).strip(), errors="coerce")
        if pd.isna(t):
            return pd.NaT
        return pd.Timestamp(
            year=1900, month=1, day=1,
            hour=t.hour, minute=t.minute, second=t.second
        )
    except Exception:
        return pd.NaT

# ===== Conversion des colonnes =====
df["_ecriture_debut_ts"] = df[col_debut].apply(convertir_heure)
df["_ecriture_fin_ts"]   = df[col_fin].apply(convertir_heure)

# ===== Calcul durée =====
def calcul_duree(row):
    debut = row["_ecriture_debut_ts"]
    fin   = row["_ecriture_fin_ts"]

    if pd.notna(debut) and pd.notna(fin):
        delta = (fin - debut).total_seconds()

        # sécurité : durée négative ou absurde
        if 0 <= delta < 3600:
            return int(delta)

    # fallback → durée moyenne
    return DUREE_MOYENNE_ECRITURE_SEC

df["duree_ecriture_sec"] = df.apply(calcul_duree, axis=1)

# ===== Diagnostic =====
nb_moyenne = (df["duree_ecriture_sec"] == DUREE_MOYENNE_ECRITURE_SEC).sum()

print("===== DURÉE ÉCRITURE CALCULÉE =====")
print(f"Total candidats : {len(df)}")
print("Le seul caniddat avec une assigantion de temps par défaut est le 0410FMS")

# ===== Nettoyage colonnes temporaires =====
df = df.drop(columns=["_ecriture_debut_ts", "_ecriture_fin_ts"])

# ===== Mise à jour dataframe principal =====
df_excel_global["duree_ecriture_sec"] = df["duree_ecriture_sec"]

display(
    df_excel_global[[
        "Numero_inclusion",
        col_debut,
        col_fin,
        "duree_ecriture_sec"
    ]]
)

===== DURÉE ÉCRITURE CALCULÉE =====
Total candidats : 134
Durées réelles calculées : 123
Durées remplacées par la moyenne (7 min) : 11


,Numero_inclusion,heure_anamnese_fin_v4,heure_rlri16imm_debut_v4,duree_ecriture_sec
0,0101CAR,2026-03-10 08:54:00,09:07:00,780
1,0102PCR,2026-03-10 09:20:00,09:30:00,600
2,0105PNR,2026-03-10 09:08:00,09:22:00,840
3,0104FJS,2026-03-10 08:44:00,08:50:00,360
4,0107DSS,2026-03-10 09:33:00,09:45:00,720
5,0109GSS,2026-03-10 09:46:00,09:56:00,600
6,0110LPR,2026-03-10 09:19:00,09:36:00,1020
7,0111MNR,2026-03-10 09:37:00,09:47:00,600
8,0112BSR,2026-03-10 10:10:00,10:25:00,900
9,0114LMS,2026-03-10 10:12:00,10:21:00,540


In [38]:
def decouper_bb_rr_par_bloc(
    df_rr,
    duree_ecriture,
    duree_bloc1,
    duree_bloc2,
    duree_bloc3,
    delta
):
    df_rr = df_rr.copy()

    # ===== Timestamp propre =====
    df_rr["Timestamp"] = pd.to_datetime(
        df_rr["Timestamp"], dayfirst=True, errors="coerce"
    )
    df_rr = df_rr.dropna(subset=["Timestamp"])

    if df_rr.empty:
        return None

    # ===== Temps de référence =====
    t0 = df_rr["Timestamp"].min()
    t_fin_rr = df_rr["Timestamp"].max()

    # ===== Bornes temporelles =====
    fin_bloc1 = t0 + pd.Timedelta(seconds=max(0, duree_bloc1 - delta))
    debut_bloc2 = fin_bloc1 + pd.Timedelta(seconds=duree_ecriture)
    fin_bloc2 = debut_bloc2 + pd.Timedelta(seconds=duree_bloc2)
    fin_bloc3 = fin_bloc2 + pd.Timedelta(seconds=duree_bloc3)

    # Sécurité (ne jamais dépasser le fichier)
    fin_bloc3 = min(fin_bloc3, t_fin_rr)

    

    return {
        "bloc1": df_rr[
            (df_rr["Timestamp"] >= t0) &
            (df_rr["Timestamp"] < fin_bloc1)
        ],
        "bloc2": df_rr[
            (df_rr["Timestamp"] >= debut_bloc2) &
            (df_rr["Timestamp"] < fin_bloc2)
        ],
        "bloc3": df_rr[
            (df_rr["Timestamp"] >= fin_bloc2) &
            (df_rr["Timestamp"] < fin_bloc3)
        ],
        "bornes": {
            "t0": t0,
            "fin_bloc1": fin_bloc1,
            "debut_bloc2": debut_bloc2,
            "fin_bloc2": fin_bloc2,
            "fin_bloc3": fin_bloc3,
            "t_fin_rr": t_fin_rr
        }
    }

In [39]:
def convertir_et_normaliser_heure(x):
    """
    Convertit n'importe quel format d'heure Excel / texte / datetime en pd.Timestamp
    normalisé sur le 1900-01-01, ne gardant que l'heure, minute, seconde.
    """
    if pd.isna(x):
        return pd.NaT

    # Excel numérique → fraction de jour ou date complète
    if isinstance(x, (int, float)):
        seconds = (float(x) * 24 * 3600) % (24*3600)
        return pd.Timestamp("1900-01-01") + pd.to_timedelta(seconds, unit="s")

    # Timestamp ou datetime → on garde juste l'heure
    if isinstance(x, (pd.Timestamp, datetime.datetime)):
        return pd.Timestamp(
            year=1900, month=1, day=1,
            hour=x.hour, minute=x.minute, second=x.second
        )

    # Texte
    try:
        t = pd.to_datetime(str(x).strip(), errors="coerce")
        if pd.isna(t):
            return pd.NaT
        return pd.Timestamp(
            year=1900, month=1, day=1,
            hour=t.hour, minute=t.minute, second=t.second
        )
    except Exception:
        return pd.NaT


In [40]:
def get_durees_blocs(pid, feuille):
    """
    Récupère les durées pour les 3 blocs et la durée d'écriture à partir de df_excel_global
    pour un PID donné et une feuille donnée.
    
    La durée d'écriture sert à décaler le début du bloc 2.
    Si la durée n'est pas renseignée ou NR, on met 7 minutes (420 s) par défaut.
    """
    pid = str(pid).upper()
    
    # ===== Bloc 1 =====
    df_b1 = pd.DataFrame(resultats_bloc1[feuille]["df"])
    ligne = df_b1.loc[df_b1["Numero_inclusion"].str.upper() == pid]
    if ligne.empty:
        raise ValueError(f"PID {pid} absent du bloc 1 dans la feuille {feuille}")
    d1 = ligne["duree_bloc1"].values[0]

    # ===== Durée écriture =====
    # On part de df_excel_global
    ligne_excel = df_excel_global.loc[df_excel_global["Numero_inclusion"].str.upper() == pid]
    if ligne_excel.empty:
        val_ecriture = 420  # défaut si absent
    else:
        val_ecriture = ligne_excel["duree_ecriture_sec"].values[0]
        if pd.isna(val_ecriture) or val_ecriture < 0:
            val_ecriture = 420  # défaut pour NR ou vide

    # ===== Bloc 2 =====
    df_b2 = pd.DataFrame(resultats_bloc2[feuille]["df"])
    d2 = df_b2.loc[df_b2["Numero_inclusion"].str.upper() == pid, "duree_bloc2"].values[0]

    # ===== Bloc 3 =====
    df_b3 = pd.DataFrame(resultats_bloc3[feuille]["df"])
    d3 = df_b3.loc[df_b3["Numero_inclusion"].str.upper() == pid, "duree_bloc3"].values[0]

    return {
        "bloc1_sec": d1 * 60,
        "duree_ecriture_sec": val_ecriture,  # durée spécifique du candidat
        "bloc2_sec": d2 * 60,
        "bloc3_sec": d3 * 60,
    }


In [41]:
# ===== Dossiers =====
input_dir = os.path.join(os.path.expanduser("~"), "Desktop", "bb_rr_tronqués_v4")
output_dir = os.path.join(os.path.expanduser("~"), "Desktop", "bb_rr_tronque_blocs_v4")
os.makedirs(output_dir, exist_ok=True)

resultats_decoupage = {}

# ===== Boucle UNIQUEMENT sur les fichiers tronqués =====
for fichier in os.listdir(input_dir):

    if not fichier.lower().endswith(".csv"):
        continue

    chemin_bb_rr = os.path.join(input_dir, fichier)

    # ===== Extraction PID depuis le nom =====
    match = re.search(r"\d{4}[A-Z]{3}", fichier.upper())
    if not match:
        print(f"⚠️ PID introuvable dans {fichier}")
        continue

    pid = match.group()

    # ===== Récupération de la feuille associée =====
    ligne = df_global.loc[df_global["Numero_inclusion"] == pid]
    if ligne.empty:
        print(f"⚠️ Feuille introuvable pour {pid}")
        continue

    feuille = ligne["Feuille"].values[0]

    print(f"\n--- Découpage {pid} ({feuille}) ---")

    # ===== Lecture BB_RR =====
    try:
        df_rr = pd.read_csv(chemin_bb_rr)
    except Exception as e:
        print(f"❌ Lecture impossible {pid} : {e}")
        continue

    # ===== Récupération des durées =====
    try:
        durees = get_durees_blocs(pid, feuille)
    except Exception as e:
        print(f"⚠️ Impossible de récupérer les durées pour {pid} ({feuille}) : {e}")
        continue

    delta_row = df_offset_bb_rr.loc[df_offset_bb_rr["Numero_inclusion"].str.upper() == pid.upper()]
    if delta_row.empty:
        delta_val = 0
    else:
        delta_val = float(delta_row["delta_retard_sec"].values[0])

    # ===== Découpage =====
    decoupe = decouper_bb_rr_par_bloc(
        df_rr=df_rr,
        duree_ecriture=durees["duree_ecriture_sec"],
        duree_bloc1=durees["bloc1_sec"],
        duree_bloc2=durees["bloc2_sec"],
        duree_bloc3=durees["bloc3_sec"],
        delta = delta_val
    )

    if decoupe is None:
        print(f"⚠️ Découpage vide pour {pid}")
        continue

    # ===== Sauvegarde par bloc =====
    for bloc in ["bloc1", "bloc2", "bloc3"]:
        df_bloc = decoupe[bloc]

        if df_bloc.empty:
            print(f"⚠️ {pid} {bloc} vide")
            continue

        nom_sortie = f"{pid}_{bloc}.csv"
        df_bloc.to_csv(
            os.path.join(output_dir, nom_sortie),
            index=False
        )

    # ===== Stockage mémoire =====
    resultats_decoupage[pid] = decoupe

    print(
        f"✔ {pid} | "
        f"B1={len(decoupe['bloc1'])} | "
        f"B2={len(decoupe['bloc2'])} | "
        f"B3={len(decoupe['bloc3'])}"
    )

print("\n✅ Découpage terminé")
print("📁 Résultats dans :", output_dir)


--- Découpage 0101CAR (APHM) ---
✔ 0101CAR | B1=20750 | B2=76071 | B3=61072

--- Découpage 0101EMS (LPC) ---
✔ 0101EMS | B1=38572 | B2=40714 | B3=48215

--- Découpage 0102EMS (LPC) ---
✔ 0102EMS | B1=9858 | B2=26786 | B3=40087

--- Découpage 0102PCR (APHM) ---
✔ 0102PCR | B1=14911 | B2=63215 | B3=43928

--- Découpage 0103BPS (LPC) ---
✔ 0103BPS | B1=31679 | B2=37500 | B3=41785

--- Découpage 0104FJS (APHM) ---
✔ 0104FJS | B1=18215 | B2=42857 | B3=42858

--- Découpage 0104IBS (LPC) ---
✔ 0104IBS | B1=31018 | B2=34286 | B3=42857

--- Découpage 0105PNR (APHM) ---
✔ 0105PNR | B1=16000 | B2=46072 | B3=50357

--- Découpage 0107DSS (APHM) ---
✔ 0107DSS | B1=25715 | B2=40714 | B3=28929

--- Découpage 0107LER (LPC) ---
✔ 0107LER | B1=21358 | B2=42857 | B3=46071

--- Découpage 0108MMS (LPC) ---
✔ 0108MMS | B1=10179 | B2=34286 | B3=47143

--- Découpage 0109GSS (APHM) ---
✔ 0109GSS | B1=10465 | B2=42857 | B3=56927

--- Découpage 0109MJR (LPC) ---
✔ 0109MJR | B1=16983 | B2=39643 | B3=47143

--- Dé

In [ ]:
## Traitment pour passer le fichier dans kubios